# EECE 5554 System Design Exercise: A Complete Sensing and Navigation System

## Learning objectives

In the labs you built and tested one sensor at a time. Real robots succeed or fail at the seams: between sensors, software, power, and the world they work in. In this exercise you design a complete sensing and navigation system for one application, on paper and on your own. You decide what the system must do and how well, choose its sensors, predict how well it will localize and navigate before anything is built, plan how the parts fit together, and work out how it will fail. Then you present the design in five slides and five minutes, as a warm up for your final project.

The goal is a way of thinking you can reuse. When someone hands you a new robot problem, you will know which questions to ask, in which order, and how to answer each one with numbers.

By the end of this exercise a well-prepared student will be able to:
<ul>
    <li> Turn an informal need into measurable requirements, each with a value, a unit, a reason, and a way to verify it </li>
    <li> Draw a system architecture and an end-to-end data pipeline, and trace every requirement to the parts that meet it </li>
    <li> Select sensors with a trade study that starts from requirements and survives a sensitivity check </li>
    <li> Predict localization error before anything is built, with an error budget and a drift analysis that use your own lab data </li>
    <li> Integrate the design on paper: frames, calibration, timing, data rates, compute, power, mass, and cost </li>
    <li> Find how the system fails, rank the failures, and design how it detects them and stays safe </li>
    <li> Present a design in five slides and five minutes, and defend it without notes </li>
</ul>

## Prerequisites

| Item | Explain |
|---|---|
| Lab 1 | Your GNSS error numbers from the stationary and walking data |
| Lab 2 | Your standalone, float, and fixed RTK precision numbers |
| Lab 3 | Your gyro noise parameters N, B, and K. Tool 4 reads `~/EECE5554/Lab3/analysis/noise_parameters.json` if it exists, and uses VN-100 datasheet values if it does not |
| Software | Python 3 with NumPy and Matplotlib, and Jupyter in VS Code, the same setup as the lab analysis notebooks. ROS 2 Jazzy on Ubuntu 24.04 is optional, for `rqt_graph`, `ros2 topic bw`, or simulation |
| Hardware | None. This is a paper design. Course hardware is optional, for checking one sensor or one assumption |

## How to use this notebook

Copy it into your own repository and work in the copy. It does not need ROS 2, so any Python 3 kernel works.

```bash
mkdir -p ~/EECE5554/SDE/analysis/figures
cp ~/5554-F26/SDE/EECE5554_SDE_instructions.ipynb ~/EECE5554/SDE/analysis/sde_analysis.ipynb
code ~/EECE5554/SDE/analysis
```

The notebook has one Part for each section of the report. Each Part says what the section must contain, gives you questions to answer on your own first, and provides tools: short functions that turn your assumptions into the numbers your report needs. Every tool starts with the numbers of the worked example, a robotic lawn mower, so every cell runs as delivered. Replace those numbers with yours as you go. A tool only helps if you can say where each of its inputs came from.

| Part | Report section | Tools |
|---|---|---|
| 0 | The method, and the scenarios to choose from | |
| 1 | Problem Definition | Tool 1: reaction time and stopping distance |
| 2 | System Architecture | |
| 3 | Sensor Selection | Tool 2: coverage and blind zones. Tool 3: trade study |
| 4 | Localization, Mapping & Navigation | Tool 4: drift. Tool 5: how often you need a fix. Tool 6: error budget |
| 5 | System Integration | Tool 7: data rates. Tool 8: power and runtime. Tool 9: parts list. Tool 10: traceability |
| 6 | Failure Analysis | Tool 11: failure modes and effects |
| 7 | The report and the five-slide talk | |

## Individual work

This exercise is individual. You may discuss course concepts with classmates, such as how an Allan deviation works or what REP 105 says, but not your scenario's design decisions, numbers, or documents. At most two students may choose the same scenario. If you share one, work independently: two designs that match beyond coincidence are an academic integrity matter.

**AI tools, the short version.** Do each section yourself first and commit your notes. Only then may you ask an AI tool to review it, and you log every use. The full rules are in *Using AI tools* near the end of this notebook. Read them before you start.

## Additional help

A complete worked example, the lawn mower whose numbers fill the tools below, is in `EECE5554_SDE_example.md`. It shows the depth expected in every section, a prototype parts list, an AI use log, and a five-slide outline. A report skeleton with every required heading and table is in `EECE5554_SDE_report_template.md`. Read the example once before you start, then put it away. Reusing its structure is expected; reusing its reasoning is not, because your scenario is different.

Run the next cell first. Every tool needs it.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Multiply a 1-sigma value by these to get a 95 percent bound.
# 1D: one axis, such as cross-track error. 2D: a horizontal radius, with equal errors on both axes.
K95 = {'1D': 1.96, '2D': 2.45}

# Save every report figure here, for example: fig.savefig(FIGURES / 'tool1_stopping.png', dpi=200)
FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

# Part 0: The method

## A design is a chain of reasons

A system design is a chain of decisions. A good one lets you point to the reason for every part: a requirement, a number, or a test. Follow the same six steps for any robot problem, in this order, and expect to loop back when a later step fails.

| Step | The question | What you produce | Report section |
|---|---|---|---|
| 1 | What must the system do, where, and how well? | Operational concept, requirements with numbers | 1. Problem Definition |
| 2 | What are the parts, and how does data flow between them? | Functional and physical architecture, pipeline figure, modes | 2. System Architecture |
| 3 | Which sensors, and why these? | Sensing needs, trade studies, placement, parts list | 3. Sensor Selection |
| 4 | How well will it know where it is, and for how long? | Methods, estimator design, error budget, drift analysis | 4. Localization, Mapping & Navigation |
| 5 | Will it all fit together? | Frames, calibration, timing, rate, power, and cost budgets, verification plan | 5. System Integration |
| 6 | How does it fail, and what happens then? | Failure modes, detection, degraded modes | 6. Failure Analysis |

```
 need → requirements → architecture → sensors → estimation → integration → failures
        ↑                                                                  │
        └──── a failed budget or an undetectable failure sends you back ───┘
```

Expect at least two loops. A runtime budget that fails sends you back to the battery or the sensors. A failure mode that nothing can detect sends you back to the architecture. Write down every loop in your report: what failed, what you changed, and why. The worked example records three.

## Four habits

1. **Numbers before names.** Write what a sensor must do, its range, rate, accuracy, and field of view, before you name a product. A requirement that says *lidar* is a solution, not a requirement.
2. **Every number has a unit, a source, and a confidence level.** A source is a datasheet, your own lab data, a calculation in this notebook, or a cited reference. A confidence level is 1σ or a 95 percent bound, stated every time.
3. **Precision is not accuracy.** The Lab 2 rule applies here too. Scatter around your own mean is precision; closeness to the truth is accuracy. An error budget needs accuracy, so say which one each number is.
4. **Design for the bad day.** Rain, a GNSS outage, a person stepping out, an old battery on a cold morning. The off-nominal cases drive a design far more than the normal mission does.

## Choose a scenario

Pick one of the twelve scenarios below and sign up at `[[ SIGNUP_LINK ]]`, first come first served, at most two students per scenario. Each scenario comes from a client who has fixed four requirements, G1 to G4. You may not relax them. Everything else is yours to decide and justify, and you will add derived requirements of your own. Where the description is silent, state an assumption and move on.

| # | Scenario | Setting | The hard part |
|---|---|---|---|
| S1 | Building mail robot | Office building, three floors, elevator | Long featureless corridors, glass, people, knowing the floor |
| S2 | Autonomous valet parking | Four-level parking garage | No GNSS inside, ramps, look-alike columns, people between cars |
| S3 | Campus sidewalk delivery | Downtown sidewalks and crosswalks | Multipath between tall buildings, curbs, crowds, rain |
| S4 | Orchard row robot | Apple orchard at night | The canopy blocks GNSS, and every row looks the same |
| S5 | Marina survey boat | Lake Union marina | Water multipath, docks and a bridge block the sky, the boat rolls |
| S6 | Warehouse inventory drone | Tall steel racks, narrow aisles | No GNSS, steel everywhere, tight mass and power limits |
| S7 | Search and rescue in smoke | Office floor during a fire | Smoke blinds cameras and lidar, no GNSS, weak radio |
| S8 | Snow-clearing robot | Driveway and sidewalk in winter | Snow hides the edges, ice, cold batteries |
| S9 | Wearable navigation aid | Bus stop to a room, for a blind pedestrian | Body-worn sensors, indoors and outdoors, a human in the loop |
| S10 | Lunar rover analog | 2 km traverse near the lunar south pole | No GNSS, no compass, harsh shadows, wheel slip |
| S11 | Construction site scanner | Building under construction | The map changes every day, stairs, 2 cm registration |
| S12 | Autonomous wheelchair | Rehabilitation hospital | Narrow doors, a passenger on board, elevators |
| S13 | Your own scenario | By approval | See below |

**Your own scenario (S13).** Send the instructor one paragraph with the mission, the setting, and four given requirements with numbers, and wait for approval before you start. It must include at least one hard sensing condition, such as a GNSS-denied or degraded segment, poor visibility, or a moving platform. It may not be the lawn mower from the worked example, and it may not be a copy of a published system.

### S1. Building mail robot

**Mission.** Collects mail at the ground-floor mail room and delivers it to 20 drop-off stations on three floors, twice a day, riding the elevator between floors.

**Setting.** Office corridors with glass walls and polished floors, doors, people walking and standing in groups, one elevator.

**Given requirements.**

- G1. Dock at each station within ±3 cm and ±3° so the tray lines up with the shelf.
- G2. Top speed 1.0 m/s in corridors.
- G3. Never touch a person, including one who steps out of a doorway 1.0 m ahead.
- G4. Know which floor it is on before it leaves the elevator, with no mistakes.

**Prototype path.** The course TurtleBot 4 in a campus building.

### S2. Autonomous valet parking

**Mission.** The driver leaves the car at the garage entrance. The car drives itself to a free space on one of four levels, parks, and comes back to the entrance when called.

**Setting.** A concrete garage with no GNSS inside, ramps between levels, rows of identical columns, dim light with bright openings, and people walking between parked cars.

**Given requirements.**

- G1. Final parked pose within ±10 cm laterally and ±1° in heading.
- G2. Speed no more than 10 km/h.
- G3. Stop for a pedestrian who steps out from between parked cars 4 m ahead.
- G4. The garage operator will paint or mount markers, but will not install cameras, beacons, or power.

**Prototype path.** The course Mustang Mach-E with its roof cameras and RTK antenna, in a campus garage. Ask the instructor before you plan around the car.

### S3. Campus sidewalk delivery robot

**Mission.** Carries food and parcels from a campus kiosk to building entrances up to 1.5 km away.

**Setting.** Downtown sidewalks between tall buildings, curbs, signalized crosswalks, pedestrians, bikes, rain, and low sun.

**Given requirements.**

- G1. All wheels stay on the sidewalk. Sidewalks are as narrow as 1.5 m, and the robot is 0.6 m wide.
- G2. Top speed 1.6 m/s.
- G3. Cross streets only at marked crosswalks, and only on the walk signal.
- G4. Stop within 2 m of the right entrance door.

**Prototype path.** A cart or handheld rig with an RTK receiver, an IMU, and a camera, pushed along a real route; the TurtleBot 4 for short tests.

### S4. Orchard row robot

**Mission.** Drives every row of a 10-hectare apple orchard each night and counts fruit with a camera.

**Setting.** Rows 3.5 m wide with trees 1 m apart and a dense canopy overhead, open headlands at both ends, uneven grass, dust, and darkness.

**Given requirements.**

- G1. Stay within ±0.15 m of the row center.
- G2. Drive at 1.5 m/s in the rows.
- G3. Know which row it is in, with no mistakes in a night, because fruit counts are filed by row.
- G4. Work 8 h a night with one recharge.

**Prototype path.** An RTK receiver and an IMU logged under tree cover on campus; a simulated orchard in Gazebo.

### S5. Marina survey boat

**Mission.** An uncrewed boat maps the water depth in a Lake Union marina so the harbor can plan dredging.

**Setting.** Open water, narrow fairways between docks, moored boats, a bridge, wind, and wakes.

**Given requirements.**

- G1. Each depth sounding located within ±0.10 m horizontally, 95 percent.
- G2. Survey lines every 5 m, including within 2 m of the docks.
- G3. 4 h on one charge.
- G4. Never touch a boat or a dock.

**Prototype path.** An RTK and IMU logging box with a low-cost single-beam echosounder, carried on a kayak; or recorded data only.

### S6. Warehouse inventory drone

**Mission.** A small drone flies along pallet racks at night and reads the label on every pallet.

**Setting.** A 100 m by 60 m warehouse with 10 m tall steel racks, aisles 2.5 m wide, no GNSS, and forklifts parked in the aisles.

**Given requirements.**

- G1. Position known within ±0.2 m whenever a label is read, so the label goes to the right bin.
- G2. Sensors and computer together weigh 300 g or less.
- G3. One full aisle per battery: 100 m long, both sides, five rack levels.
- G4. Stay at least 0.5 m from racks and forklifts.

**Prototype path.** A handheld sensor rig carried along a shelf, or the same rig on a TurtleBot 4 to stand in for one rack level.

### S7. Search and rescue in smoke

**Mission.** A tracked robot enters a smoke-filled office floor ahead of firefighters, maps it, reports where people are, and comes back to the door.

**Setting.** Smoke with visibility under 1 m, no lights, debris, open and closed doors, sprinkler water, no GNSS, and weak radio.

**Given requirements.**

- G1. Report each person's location within ±1 m on the floor plan.
- G2. Back at the entry door within 30 min.
- G3. Keep working through radio loss of up to 5 min.
- G4. Operate in air up to 60 °C.

**Prototype path.** The TurtleBot 4 in a dark room; the effect of smoke on each sensor from published data sets.

### S8. Snow-clearing robot

**Mission.** Clears a 60 m driveway and 200 m of sidewalk for one house after each snowfall, overnight.

**Setting.** Snow over the curbs and the paint, ice, falling snow, temperatures down to -15 °C, and trees and houses next to the sidewalk.

**Given requirements.**

- G1. Plow edge within ±0.10 m of the sidewalk edge.
- G2. Finish within 3 h.
- G3. Work at -15 °C.
- G4. Stop for a person or a car 1.5 m ahead.

**Prototype path.** An RTK receiver, an IMU, and wheel odometry on a cart, on a wet or slippery surface; cold derating from Lecture 06.

### S9. Wearable navigation aid for a blind pedestrian

**Mission.** A body-worn system guides a blind user from a bus stop to a room inside a building, with turn cues by vibration, and warns of obstacles from the ground up to head height.

**Setting.** A sidewalk, a building entrance, a lobby, an elevator or stairs, corridors, open doors, and low branches.

**Given requirements.**

- G1. Warn of any obstacle from the ground to head height within 3 m, at a walking speed of 1.4 m/s.
- G2. Arrive within 1 m of the room door.
- G3. All worn parts together weigh under 300 g and run 12 h on one charge.
- G4. The user never needs to look at or hold a phone.

**Prototype path.** A VN-100 on the shoe for step detection, a chest camera, and a phone for GNSS.

### S10. Lunar rover analog

**Mission.** A small rover drives 2 km from its lander to a crater rim near the lunar south pole and back, with little help from Earth.

**Setting.** No GNSS, no global magnetic field, the sun near the horizon with long and moving shadows, loose regolith, slopes up to 15°, and a few seconds of radio delay to Earth.

**Given requirements.**

- G1. Return to within 5 m of the lander.
- G2. Drive at least 300 m between operator check-ins.
- G3. Average power under 40 W, from solar panels.
- G4. Climb and descend slopes up to 15°.

**Prototype path.** The TurtleBot 4 in a dark room with one lamp as the sun; a sand box for wheel slip; simulation.

### S11. Construction site scanner

**Mission.** A legged robot walks a building under construction every evening, scans it, and reports every wall or column more than 2 cm from the design model.

**Setting.** Concrete floors, stairs, temporary walls, equipment that moves every day, dust, and GNSS only outside the building.

**Given requirements.**

- G1. Scans registered to the design model within ±2 cm, 95 percent.
- G2. One 1,500 m² floor per hour.
- G3. Climbs the stairs between floors.
- G4. The site crew allows at most 10 fixed markers per floor.

**Prototype path.** A handheld camera or lidar rig with an IMU; the TurtleBot 4 on one floor.

### S12. Autonomous wheelchair

**Mission.** Takes a seated patient from their room to the therapy gym and back, on request.

**Setting.** Hospital corridors, 0.9 m doorways, elevators, a ramp, staff, other wheelchairs, and beds in the hall.

**Given requirements.**

- G1. Pass through 0.9 m doorways. The chair is 0.68 m wide.
- G2. Accelerate and brake at no more than 1.0 m/s², or 2.0 m/s² in an emergency stop.
- G3. Stop for a person who steps out 1.5 m ahead.
- G4. The patient can stop the chair at any time with one button.

**Prototype path.** The TurtleBot 4 as a scaled stand-in for the drive base, with doorway mock-ups.

# Part 1: Problem Definition

## 1.1 Operational concept

Before any requirement, describe how the system will be used. One page is enough.
<ul>
    <li> <b>Users and stakeholders:</b> who uses it, who maintains it, who could be hurt by it, and what each of them cares about </li>
    <li> <b>One normal mission</b>, start to finish: where it starts, what it does, and how it knows it is done </li>
    <li> <b>The operating envelope:</b> indoors or outdoors, day or night, weather, temperature, surfaces, speeds, and how many people are nearby </li>
    <li> <b>At least two off-nominal events</b>, such as a sensor failing, the GNSS dropping out, rain, or a person stepping in the way, and what the system should do in each </li>
    <li> <b>Scope:</b> what your design does not cover, stated plainly </li>
</ul>

### Pass 1, on your own

Answer these in `design_journal.md` before you open any AI tool, then commit the journal.

1. Who is the client, and what would make them say the system works?
2. Walk through one normal mission. What must the system sense at each step?
3. List ten ways the world can be worse than normal. Which three worry you most, and why?
4. What will you leave out of scope?

## 1.2 Requirements

A requirement is a testable promise. Write at least eight: the four given ones, and at least three derived ones that come from your own analysis, with the math shown. Use this table. The rows below are from the worked example.

| ID | Requirement | Value | Type | Source or rationale | Verification |
|---|---|---|---|---|---|
| R1 | The mower's body stays inside the lawn boundary | at most 0.10 m outside, 95 percent of boundary passes | Given | Client | Test: 20 laps of a taped boundary, RTK rover as ground truth |
| R4 | Cross-track localization error on boundary passes | at most 0.035 m, 1σ | Derived from R1 | Error budget, Tool 6 | Analysis, then the R1 test |
| R6 | Detect a 0.3 m tall object ahead of the mower | at 0.6 m or more | Derived from R5 | Stopping distance, Tool 1 | Test: 20 trials with a foam block |

Rules for a good requirement:
<ul>
    <li> <b>What, not how.</b> Detect a 0.3 m tall object at 0.6 m, not use a lidar </li>
    <li> <b>One idea each.</b> Split any requirement that needs the word <i>and</i> </li>
    <li> <b>A number, a unit, and a condition or a confidence level:</b> 95 percent of samples over a 30 minute run, on wet grass, at night </li>
    <li> <b>Verifiable</b> by test, analysis, inspection, or demonstration, named in the last column </li>
    <li> <b>Traceable:</b> given by the client, or derived from another requirement with the math shown </li>
</ul>

Cover at least these kinds: mission performance (coverage, time, accuracy), safety (stopping, the safe state), environment (light, weather, temperature, surfaces), endurance (runtime, recharging), and constraints (cost, mass, size, interfaces).

## 1.3 Flow-down: from do not hit anyone to sensor numbers (Tool 1)

The most useful derived requirements come from safety. A robot at speed $v$ needs time to notice an object and distance to brake. With reaction time $t$ and braking deceleration $a$:

$$d_{stop} = v\,t + \frac{v^2}{2a} \qquad d_{detect} \ge d_{stop} + d_{margin}$$

The reaction time is a latency budget. In the worst case the object appears just after a sensor frame, so a full sensor period passes before anything happens. Then perception, planning and control, and the actuators each take their share:

$$t = \frac{1}{f_{sensor}} + t_{perception} + t_{planning} + t_{actuation}$$

Solve the first equation the other way to get the highest safe speed for a sensor that only sees so far:

$$v_{max} = -a\,t + \sqrt{(a\,t)^2 + 2a\,(d_{detect} - d_{margin})}$$

Use the worst surface in your operating concept for $a$, not the best. Wet grass, ice, and a passenger who must not be thrown forward all lower it.

In [ ]:
def latency_budget(sensor_rate_hz, perception_s, planning_s, actuation_s):
    """Worst-case reaction time in s: one full sensor period plus every processing stage."""
    parts = {'sensor period': 1.0 / sensor_rate_hz,
             'perception': perception_s,
             'planning and control': planning_s,
             'actuation': actuation_s}
    return sum(parts.values()), parts


def stopping_distance(v, reaction_s, decel):
    """Distance in m covered from the moment an object appears until the robot stands still."""
    return v * reaction_s + v ** 2 / (2.0 * decel)


def max_safe_speed(detect_range, margin, reaction_s, decel):
    """Highest speed in m/s that stops margin m short of an object first seen detect_range m away."""
    d = detect_range - margin
    if d <= 0:
        return 0.0
    return -decel * reaction_s + np.sqrt((decel * reaction_s) ** 2 + 2.0 * decel * d)

In [ ]:
# Example values: the lawn mower in EECE5554_SDE_example.md. Replace them with yours.
V_TOP = 0.5      # m/s, top mowing speed
DECEL = 0.5      # m/s², braking on wet grass, the worst surface in the operating concept
MARGIN = 0.15    # m, the mower must stand still this far short of the object (requirement R5)

t_react, parts = latency_budget(sensor_rate_hz=10.0, perception_s=0.10,
                                planning_s=0.05, actuation_s=0.10)
d_stop = stopping_distance(V_TOP, t_react, DECEL)
for name, t in list(parts.items()) + [('total reaction time', t_react)]:
    print(f'{name:22s} {1000 * t:5.0f} ms')
print(f'Stopping distance at {V_TOP} m/s: {d_stop:.3f} m')
print(f'Required detection range: {d_stop + MARGIN:.3f} m, the stopping distance plus the {MARGIN} m margin')
for r in (0.4, 0.6, 1.0, 2.0):
    print(f'A sensor that sees {r:.1f} m allows at most {max_safe_speed(r, MARGIN, t_react, DECEL):.2f} m/s')

v = np.linspace(0.05, 2.0, 200)
fig, ax = plt.subplots(figsize=(7, 4))
for a, surface in ((1.0, 'dry'), (0.5, 'wet'), (0.25, 'slippery')):
    ax.plot(v, stopping_distance(v, t_react, a) + MARGIN, label=f'{surface}, {a} m/s²')
ax.axvline(V_TOP, color='gray', ls='--', lw=0.8, label='top speed')
ax.set_xlabel('speed (m/s)')
ax.set_ylabel('required detection range (m)')
ax.set_title(f'Detection range needed with a {1000 * t_react:.0f} ms reaction time')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

**In your report.** Give the latency budget, the stopping distance at top speed on the worst surface, and the detection range it implies, as a derived requirement. If no affordable sensor sees that far, the answer is a lower top speed, and `max_safe_speed` tells you how much lower. Do the same for any other safety distance in your scenario: a doorway, a curb, a drop-off, a rack.

# Part 2: System Architecture

## 2.1 Functional architecture: what the system does

Start from functions, not parts. Most sensing and navigation systems share one backbone: sensors feed drivers that stamp and convert their data, an estimator turns the data into a pose, perception turns it into obstacles and a map, a planner and a controller decide what to do, and actuators do it. A safety supervisor watches every block. This is the generic pipeline:

```
 SENSORS             DRIVERS                ESTIMATION             DECISION AND ACTION
 GNSS            ──► GNSS driver     ──┐
 IMU             ──► IMU driver      ──┼──► state estimator ──┐
 wheel encoders  ──► base driver     ──┘                      ├──► planner ──► controller ──► motors
 depth camera    ──► camera driver   ─────► obstacle detector─┘
 safety supervisor: watches every block, owns the modes, and can stop the motors without the main computer
```

Your **pipeline figure** must show every sensor, every processing step, the rate of each data flow, and where the safety supervisor can stop the robot. Make it one figure that is readable at slide size; it becomes Slide 2. Draw it in draw.io or Mermaid, not by hand.

## 2.2 Physical architecture: what it is made of

Say which computer runs which software, which buses and links connect the parts (USB, Ethernet, CAN, UART, I2C, Wi-Fi, cellular), how power is distributed, and where outside data comes from, such as RTK corrections over NTRIP as in Lab 2, or a map. Say what runs on a microcontroller instead of Linux. Put each safety function where a crashed main computer cannot stop it from working.

## 2.3 Operating modes

Give a table or a state diagram with at least a normal mode, one degraded mode, and a safe stop, each with the condition that enters it and the condition that leaves it. Take the thresholds from Parts 4 and 6, so they are numbers, not words. A row from the worked example:

| Mode | Entered when | Behavior | Left when |
|---|---|---|---|
| Degraded position | the 95 percent position bound passes 0.07 m | interior passes only, 0.3 m/s | the bound stays under 0.07 m for 5 s |

## 2.4 The ROS 2 view

List the nodes, the topics with their message types and rates, and the frames. Use standard messages where they exist: `sensor_msgs/NavSatFix`, `sensor_msgs/Imu`, `nav_msgs/Odometry`, `sensor_msgs/LaserScan`, `sensor_msgs/PointCloud2`, `sensor_msgs/Range`. Name the package behind each node: one of your own Lab 1 to 3 drivers, or an existing package. Check that every package you name exists for Jazzy, with `apt search ros-jazzy-<name>` or at https://index.ros.org. Packages that systems like yours are often built on:

| Package | What it does |
|---|---|
| robot_localization | EKF and UKF fusion of odometry, IMU, and GNSS, with navsat_transform to bring GNSS into the map frame |
| Nav2 | Navigation: planners, controllers, behavior trees, and costmaps |
| slam_toolbox | 2D lidar SLAM, and localization in a saved map |
| rtabmap_ros | Visual and lidar SLAM with loop closure |
| tf2_tools | `ros2 run tf2_tools view_frames` draws your TF tree |
| rosbag2 | Recording and replaying data, which your verification plan will need |

### Pass 1, on your own

1. Which block is a single point of failure?
2. Where does the system decide that it is unsafe, and what does it do then?
3. Which functions must keep working when the main computer crashes?
4. Which block needs the most compute, and where does it run?

# Part 3: Sensor Selection

## 3.1 From requirements to sensing needs

Before you name any product, write a sensing needs table: each need, the requirement it comes from, the quantity that matters, and the value. Rows from the worked example:

| Need | From | Quantity | Value |
|---|---|---|---|
| See a 0.3 m tall object in the path | R5, Tool 1 | detection range | 0.6 m or more |
| Cover the full width of the path | R5, Tool 2 | horizontal field of view | 64° or more |
| Absolute position on boundary passes | R4 | 1σ error | 0.035 m or less |
| Heading through a 15 m gap in GNSS | R8, Tools 4 and 5 | gyro N and B | VN-100 class or better |

## 3.2 What each kind of sensor gives you, and how it fails

| Sensor | Measures | Good at | Degrades or fails when |
|---|---|---|---|
| GNSS, standalone | Absolute position and time | Global, cheap, no drift | The sky is blocked; multipath near buildings and water; meter-level error |
| RTK GNSS | Absolute position to centimeters | Boundary-level accuracy outdoors | The corrections link drops; a canopy pushes it to float; wrong fixes near reflectors |
| Gyroscope | Angular rate | High rate, needs no outside signal | Integrated heading drifts with N, B, and K from Lab 3 |
| Accelerometer | Gravity plus acceleration | Tilt, vibration, impacts | Integrated twice, position drifts away in seconds |
| Magnetometer | Magnetic field, so heading | Absolute heading for little power | Steel, motors, and currents nearby; indoors |
| Wheel encoders | Wheel rotation | Smooth short-term motion | Slip on wet or loose ground, wheel wear, scale error |
| 2D lidar | Ranges in one plane | Accurate geometry, works in the dark | Misses anything above or below its plane; glass, black surfaces, rain |
| 3D lidar | Ranges in many planes | Rich geometry for mapping | Cost, power, and data rate; rain, fog, dust |
| Stereo or depth camera | Dense depth and color | Shape and meaning at short range | Depth error grows with range squared; low texture, darkness, sun on active infrared |
| Monocular camera | Color images | Meaning, markers, lane lines | No scale; lighting changes |
| Radar | Range and radial speed | Works in fog, smoke, dust, and rain | Low resolution, multipath |
| Ultrasonic | Range to the nearest echo | Cheap, short range, any lighting | Wide beam, soft surfaces, crosstalk between sensors |
| Thermal camera | Temperature image | People and animals in darkness or smoke | Low resolution, cost |
| UWB | Range to fixed anchors | Indoor absolute position | Anchors to install; bias when the line of sight is blocked |
| Barometer | Pressure, so altitude | Floor changes | Weather, doors, and ventilation change the pressure |
| Bumper, cliff, lift, tilt | Contact and orientation events | The last line of safety | They act only once the event has begun |

Notice the last column. Two sensors that fail under the same condition, such as two cameras in fog, give you no protection against that condition. Pair sensors whose physics differ.

## 3.3 Field of view, blind zones, and resolution (Tool 2)

A sensor mounted at height $h$, tilted down by $\theta$, with vertical field of view $\varphi$:

- first sees the ground at $d_{min} = h / \tan(\theta + \varphi/2)$; everything nearer on the ground is a blind zone
- at distance $d$ sees heights from $h - d\tan(\theta + \varphi/2)$ up to $h - d\tan(\theta - \varphi/2)$
- needs a horizontal field of view of $2\arctan(w / 2d)$ to cover a path $w$ wide at distance $d$
- puts about $s / (d\,\Delta\alpha)$ points or pixels on a target of width $s$, for angular resolution $\Delta\alpha$. Ask for at least three

A stereo camera with focal length $f$ in pixels, baseline $B$, and disparity error $\sigma_d$, about 0.1 to 0.3 px, has a depth error that grows with the square of the range $z$:

$$\sigma_z = \frac{z^2\,\sigma_d}{f\,B}$$

Check your datasheet against this formula. If they disagree, find out why before you trust either.

In [ ]:
def vertical_window(height, pitch_down_deg, vfov_deg, d):
    """Lowest and highest heights in m that a sensor sees at horizontal distance d m.

    height is the mounting height in m. pitch_down_deg tilts the optical axis below horizontal.
    """
    lower = np.radians(pitch_down_deg + vfov_deg / 2.0)   # lower edge of the view, below horizontal
    upper = np.radians(pitch_down_deg - vfov_deg / 2.0)   # upper edge; negative points above horizontal
    d = np.asarray(d, dtype=float)
    return np.maximum(height - d * np.tan(lower), 0.0), height - d * np.tan(upper)


def ground_blind_zone(height, pitch_down_deg, vfov_deg):
    """Distance in m from the sensor to the first point on the ground it can see."""
    lower = np.radians(pitch_down_deg + vfov_deg / 2.0)
    return height / np.tan(lower) if lower > 0 else np.inf


def hfov_needed(width, d):
    """Horizontal field of view in degrees that covers a path width m wide at distance d m."""
    return np.degrees(2.0 * np.arctan(width / (2.0 * d)))


def points_on_target(target_width, d, angular_res_deg):
    """Number of scan points or pixels that land on a target target_width m wide at d m."""
    return target_width / (d * np.radians(angular_res_deg))


def stereo_depth_sigma(z, focal_px, baseline_m, disparity_sigma_px=0.2):
    """1-sigma depth error in m of a stereo camera at range z m."""
    return np.asarray(z, dtype=float) ** 2 * disparity_sigma_px / (focal_px * baseline_m)

In [ ]:
# Example: an OAK-D Lite on the mower, 0.25 m above the grass and tilted 15 degrees down.
# Its stereo pair, from the Luxonis datasheet: 640 x 480 pixels, HFOV 73 deg, VFOV 58 deg, baseline 0.075 m.
H_CAM, PITCH, VFOV, HFOV, WIDTH_PX = 0.25, 15.0, 58.0, 73.0, 640
FOCAL_PX = (WIDTH_PX / 2) / np.tan(np.radians(HFOV / 2))
PATH_WIDTH = 0.75     # m, the mower is 0.55 m wide, plus 0.10 m on each side
D_DETECT = 0.6        # m, from Tool 1
OBJECT_H = 0.30       # m, the smallest object in requirement R5

low, high = vertical_window(H_CAM, PITCH, VFOV, D_DETECT)
print(f'Ground first visible at {ground_blind_zone(H_CAM, PITCH, VFOV):.2f} m')
print(f'At {D_DETECT} m the camera sees heights from {low:.2f} m to {high:.2f} m')
print(f'HFOV needed at {D_DETECT} m: {hfov_needed(PATH_WIDTH, D_DETECT):.1f} deg; the camera has {HFOV} deg')
print(f'HFOV needed at 0.4 m, its minimum depth: {hfov_needed(PATH_WIDTH, 0.4):.1f} deg')
print(f'Focal length {FOCAL_PX:.0f} px; stereo depth error at 3 m: {100 * stereo_depth_sigma(3.0, FOCAL_PX, 0.075):.1f} cm (1σ)')
print(f'Points on a 5 cm post at 3 m: camera {points_on_target(0.05, 3.0, HFOV / WIDTH_PX):.1f}, '
      f'2D lidar with 0.72 deg steps {points_on_target(0.05, 3.0, 0.72):.1f}')

d = np.linspace(0.0, 2.0, 201)
low, high = vertical_window(H_CAM, PITCH, VFOV, d)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.fill_between(d, low, high, alpha=0.25, label='what the camera sees')
ax.add_patch(plt.Rectangle((D_DETECT, 0.0), 0.15, OBJECT_H, color='tab:red', alpha=0.6, label='0.3 m object'))
ax.plot(0.0, H_CAM, 'ks', label='camera')
ax.axhline(0.0, color='green', lw=2)
ax.set_xlim(0.0, 2.0)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel('distance ahead (m)')
ax.set_ylabel('height (m)')
ax.set_title('Side view of the camera coverage')
ax.grid(alpha=0.3)
ax.legend(loc='upper left')
plt.show()

## 3.4 Trade study (Tool 3)

Do a trade study for each key sensing function, at least two of them, such as absolute positioning and obstacle detection. Follow these steps in order.

1. **Candidates.** At least three for each function, from different physics where possible. A different approach, such as markers instead of a sensor, counts as a candidate.
2. **Must-meet screen.** Thresholds that come straight from your requirements. A candidate that misses any of them is out, whatever its other strengths, and you record why.
3. **Criteria and weights.** The weights come from your requirements and your client's priorities, and they add up to 1. Write one sentence of reasoning for each weight **before** you score any candidate. Leave out criteria that do not tell the candidates apart, and say why.
4. **Scores.** The tool maps each value to a score from 0 at the threshold to 1 at your goal, so a score means something even when you add a candidate later. Some criteria have no datasheet number, such as how well a sensor tells grass from a pet. Give those a judgment score from 0 to 1, and write one sentence for each.
5. **Sensitivity.** The tool multiplies every weight by a random factor, with about 25 percent spread, thousands of times, and counts how often each candidate wins. If your winner wins less than about 70 percent of the time, the study does not support a clear choice. Say so, and decide on another basis, such as a test.
6. **Decide.** Name the winner, the runner-up, and the condition that would make you switch.

In [ ]:
def score_value(x, better, threshold, goal):
    """0 at the threshold, 1 at the goal, linear in between, clipped to the range 0 to 1."""
    if better == 'higher':
        s = (x - threshold) / (goal - threshold)
    else:
        s = (threshold - x) / (threshold - goal)
    return float(np.clip(s, 0.0, 1.0))


def meets(x, better, threshold):
    """True when x is on the right side of a must-meet threshold."""
    return x >= threshold if better == 'higher' else x <= threshold


def trade_study(candidates, criteria, n_trials=5000, spread=0.25, seed=0, title='Trade study'):
    """Must-meet screen, weighted scores, and a check of how much the winner depends on the weights.

    candidates: {name: {attribute: value}}
    criteria: list of dicts with attr, weight, better ('higher' or 'lower'), threshold, goal, and
              must (True when the threshold is a must-meet requirement). The weights add up to 1.
    """
    w = np.array([c['weight'] for c in criteria], dtype=float)
    if not np.isclose(w.sum(), 1.0):
        raise ValueError(f'The weights add up to {w.sum():.3f}, not 1')
    names, rows = [], []
    for name, attrs in candidates.items():
        failed = [c['attr'] for c in criteria
                  if c['must'] and not meets(attrs[c['attr']], c['better'], c['threshold'])]
        if failed:
            reasons = ', '.join(failed)
            print(f'Screened out: {name}, fails {reasons}')
            continue
        names.append(name)
        rows.append([score_value(attrs[c['attr']], c['better'], c['threshold'], c['goal'])
                     for c in criteria])
    if not names:
        print('No candidate meets every must-meet threshold. Revisit the candidates or the requirements.')
        return None
    S = np.array(rows)
    total = S @ w
    rng = np.random.default_rng(seed)
    wins = np.zeros(len(names))
    for _ in range(n_trials):
        wt = w * np.exp(spread * rng.standard_normal(w.size))
        wins[np.argmax(S @ (wt / wt.sum()))] += 1
    order = np.argsort(-total)
    width = max(len(n) for n in names)
    print()
    print('Candidate'.ljust(width) + '   score   wins')
    for i in order:
        print(f'{names[i]:{width}s}   {total[i]:5.3f}  {100 * wins[i] / n_trials:4.0f}%')
    share = wins[order[0]] / n_trials
    verdict = 'a clear choice' if share >= 0.7 else 'not a clear choice: decide with a test or a new criterion'
    print(f'Winner: {names[order[0]]}, in {100 * share:.0f}% of {n_trials} weight perturbations: {verdict}')

    fig, ax = plt.subplots(figsize=(9, 0.6 * len(names) + 1.6))
    left = np.zeros(len(names))
    for j, c in enumerate(criteria):
        if c['weight'] > 0:
            part = S[order, j] * w[j]
            ax.barh(np.arange(len(names)), part, left=left, label=c['attr'])
            left += part
    ax.set_yticks(np.arange(len(names)))
    ax.set_yticklabels([names[i] for i in order])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('weighted score (1 would be ideal on every criterion)')
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    fig.tight_layout()
    plt.show()
    return {names[i]: {'score': float(total[i]), 'win_share': float(wins[i] / n_trials)} for i in order}

In [ ]:
# Example: obstacle detection on the mower. Numbers from the Luxonis, Slamtec, and DFRobot datasheets
# and their list prices; the judgment scores (0 to 1) are explained in the worked example.
candidates = {
    'Depth camera, OAK-D Lite': {
        'range_m': 3.0, 'min_range_m': 0.40, 'hfov_deg': 73, 'cost_usd': 269,
        'low_objects': 1.0, 'outdoors': 0.6, 'grass_vs_obstacle': 1.0, 'ros2_driver': 1.0},
    '2D lidar, RPLIDAR C1': {
        'range_m': 6.0, 'min_range_m': 0.05, 'hfov_deg': 360, 'cost_usd': 80,
        'low_objects': 0.2, 'outdoors': 0.4, 'grass_vs_obstacle': 0.3, 'ros2_driver': 1.0},
    'Ultrasonic array, 3 x A02YYUW': {
        'range_m': 2.0, 'min_range_m': 0.03, 'hfov_deg': 150, 'cost_usd': 48,
        'low_objects': 0.7, 'outdoors': 0.9, 'grass_vs_obstacle': 0.1, 'ros2_driver': 0.7},
    'Bumper switch only': {
        'range_m': 0.0, 'min_range_m': 0.0, 'hfov_deg': 180, 'cost_usd': 10,
        'low_objects': 1.0, 'outdoors': 1.0, 'grass_vs_obstacle': 0.0, 'ros2_driver': 1.0},
}
criteria = [
    {'attr': 'range_m',           'weight': 0.15, 'better': 'higher', 'threshold': 0.6, 'goal': 4.0,  'must': True},
    {'attr': 'min_range_m',       'weight': 0.00, 'better': 'lower',  'threshold': 0.5, 'goal': 0.05, 'must': True},
    {'attr': 'hfov_deg',          'weight': 0.10, 'better': 'higher', 'threshold': 64,  'goal': 180,  'must': True},
    {'attr': 'cost_usd',          'weight': 0.15, 'better': 'lower',  'threshold': 300, 'goal': 20,   'must': True},
    {'attr': 'low_objects',       'weight': 0.20, 'better': 'higher', 'threshold': 0.0, 'goal': 1.0,  'must': False},
    {'attr': 'outdoors',          'weight': 0.15, 'better': 'higher', 'threshold': 0.0, 'goal': 1.0,  'must': False},
    {'attr': 'grass_vs_obstacle', 'weight': 0.20, 'better': 'higher', 'threshold': 0.0, 'goal': 1.0,  'must': False},
    {'attr': 'ros2_driver',       'weight': 0.05, 'better': 'higher', 'threshold': 0.0, 'goal': 1.0,  'must': False},
]
obstacle_study = trade_study(candidates, criteria, title='Mower obstacle detection')

**In your report.** For each trade study: the must-meet screen with the reason each screened candidate failed, the weights with their reasons, the scores with datasheet citations, the bar chart, the sensitivity result, and your decision with the runner-up. A study whose weights were chosen to make a favorite win shows up in the sensitivity check, and in the questions after your talk.

## 3.5 Placement, and the parts list for a prototype

Show where every sensor goes in a top view and a side view, with its field of view and blind zones from Tool 2. Then list the parts of a prototype you could build in the final project with Tool 9 in Part 5: model, quantity, price, mass, power, source, and whether the course owns it.

Course hardware you can plan around:
<ul>
    <li> TurtleBot 4: a Create 3 base with an IMU, wheel encoders, an optical floor sensor, bumpers, and cliff sensors; an RPLIDAR A1M8; an OAK-D camera; a Raspberry Pi 4B. Top speed about 0.3 m/s </li>
    <li> SparkFun RTK mosaic-X5 receivers with NTRIP corrections, from Lab 2 </li>
    <li> VectorNav VN-100 IMUs, from Lab 3 </li>
    <li> The Mustang Mach-E with its roof cameras and RTK antenna. Ask the instructor before you plan around it </li>
</ul>

Course-owned parts cost you nothing, and cost is a design criterion like any other. A prototype that tests your riskiest assumption cheaply beats an expensive one that tests everything a little.

### Pass 1, on your own

1. For each sensing need, what is the cheapest sensor that could possibly meet it?
2. Which need is hardest to meet, and what happens to the design if you cannot meet it?
3. Which two of your sensors fail under the same condition?
4. Where does each sensor go, and what can it not see from there?

# Part 4: Localization, Mapping & Navigation

## 4.1 Choose your methods, and say why

For each of the three functions below, name your choice, one alternative you considered, and why the alternative lost.

| Function | Common choices |
|---|---|
| Localization | GNSS or RTK fused with an IMU and odometry in an EKF; scan matching against a saved map (AMCL, or slam_toolbox in localization mode); visual or visual-inertial odometry; fiducial markers or UWB anchors; pedestrian dead reckoning with zero-velocity updates; sun sensors or star trackers where nothing else exists |
| Mapping | No map, only a taught route; a topological graph of places; a 2D occupancy grid; a 2.5D elevation map; a 3D voxel or point cloud map; a prior map such as a floor plan, a design model, or OpenStreetMap. Say how the map is built, stored, and kept current |
| Navigation | Global planners: `A*`, coverage patterns for mowing or surveying, lattice planners for cars. Local planners and controllers: DWB, MPPI, regulated pure pursuit, Stanley, MPC. Mission logic: behavior trees or state machines. The final approach or docking |

Use the frames of REP 105. `map` is globally consistent but can jump when a fix arrives; `odom` is smooth but drifts; `base_link` is the robot. Run control loops in `odom` and set mission goals in `map`.

## 4.2 Estimator design

Give a table with the state vector, the inputs to the process model, each measurement with its rate and 1σ noise, the frames, how the filter starts, and how it rejects bad data. Two rules from the labs apply. Parse the GNSS fix quality, never infer it, and use it to set the measurement noise or to reject the fix. And gate every measurement: one whose innovation fails a chi-square test at 99 percent is rejected and counted, and a long run of rejections is itself a failure to detect.

## 4.3 Where your absolute fixes come from

GNSS horizontal error is roughly $\mathrm{HDOP} \times \sigma_{UERE}$, from the GNSS lectures. Start from your own lab data:

| Source | Typical 1σ horizontal | Where your number comes from |
|---|---|---|
| Standalone GNSS, open sky | about 1.5 m | Lab 1 and Lab 2 |
| RTK float | about 0.4 m | Lab 2 |
| RTK fixed | about 0.02 m | Lab 2 |
| Anything else: markers, UWB, scan matching | | A datasheet, a paper, or a test you plan |

These Lab 2 values are precision: scatter around your own mean. Near walls, water, and trees the mean itself moves, so accuracy is worse. Say which one you use.

## 4.4 Drift between fixes (Tool 4)

Between absolute fixes the robot dead-reckons, and its error grows. For a ground robot that drives straight at speed $v$ and integrates its gyro for heading, each error source grows at its own rate:

| Source | 1σ cross-track error after time $t$ |
|---|---|
| Heading error at the start, $\sigma_{\psi 0}$ | $v\,\sigma_{\psi 0}\,t$ |
| Gyro bias left after calibration, $B$ | $v\,B\,t^2/2$ |
| Angle random walk, $N$ | $v\,N\sqrt{t^3/3}$ |
| Rate random walk, $K$ | $v\,K\sqrt{t^5/20}$ |
| Lateral wheel slip, $q$ in m²/s | $\sqrt{q\,t}$ |

Along the track, a wheel odometry scale error $\varepsilon$ adds about $\varepsilon\,v\,t$. The sources are independent, so the total is their root sum square. Notice the powers of $t$: over short gaps the starting heading error dominates, and over long gaps the bias always wins. This is the Lab 3 Question 5 result, carried from heading to position.

An IMU alone, with no wheels, is much worse, because a tilt error lets gravity leak into the horizontal acceleration:

| Source | 1σ position error after time $t$ |
|---|---|
| Velocity error at the start, $\sigma_{v0}$ | $\sigma_{v0}\,t$ |
| Accelerometer bias, $b_a$ | $b_a\,t^2/2$ |
| Accelerometer noise, the velocity random walk | $\mathrm{VRW}\sqrt{t^3/3}$ |
| Tilt error at the start, $\sigma_{\theta 0}$ | $g\,\sigma_{\theta 0}\,t^2/2$ |
| Gyro bias, through tilt | $g\,B\,t^3/6$ |
| Gyro noise, through tilt | $g\,N\sqrt{t^5/20}$ |

These short-time formulas leave out Earth rotation and the Schuler effect, which matter only after many minutes. Run the cells below to see what they mean for a VN-100: seconds, not minutes. Drones and wearables need something more, such as visual-inertial odometry, or a zero-velocity update at every step for an IMU worn on the foot.

In [ ]:
def gyro_noise(path='~/EECE5554/Lab3/analysis/noise_parameters.json', axis='z'):
    """N, B, and K of one gyro axis in SI units: your Lab 3 results if they exist, else the VN-100 datasheet."""
    p = Path(path).expanduser()
    if p.exists():
        a = json.loads(p.read_text())[axis]
        return {'N': a['N'], 'B': a['B'], 'K': a.get('K') or 0.0, 'source': f'Lab 3, {p}'}
    return {'N': np.radians(0.0035),          # 0.0035 deg/s/sqrt(Hz) noise density
            'B': np.radians(10.0) / 3600.0,   # under 10 deg/h in-run bias stability
            'K': 0.0,                         # not on the datasheet
            'source': 'VN-100 datasheet'}


def dead_reckoning_drift(t, v, N, B, K=0.0, sigma_y0=0.0, sigma_psi0=0.0, q_lat=0.0, scale_err=0.0):
    """1-sigma error growth for a ground robot that drives straight and integrates its gyro for heading.

    t in s, v in m/s, N in rad/sqrt(s), B in rad/s, K in rad/s/sqrt(s), sigma_y0 in m, sigma_psi0 in rad,
    q_lat in m^2/s for lateral wheel slip, and scale_err as a fraction of the distance driven.
    Returns the cross-track terms (a dict of arrays in m), their root sum square, and the along-track error.
    """
    t = np.asarray(t, dtype=float)
    terms = {'start position': np.full_like(t, sigma_y0),
             'start heading': v * sigma_psi0 * t,
             'gyro bias B': v * B * t ** 2 / 2.0,
             'angle random walk N': v * N * np.sqrt(t ** 3 / 3.0),
             'rate random walk K': v * K * np.sqrt(t ** 5 / 20.0),
             'lateral slip': np.sqrt(q_lat * t)}
    cross = np.sqrt(sum(x ** 2 for x in terms.values()))
    along = np.sqrt(sigma_y0 ** 2 + (scale_err * v * t) ** 2)
    return terms, cross, along


def inertial_only_drift(t, N, B, vrw, accel_bias, sigma_v0=0.0, sigma_tilt0=0.0, g=9.81):
    """1-sigma horizontal position error of an IMU with no aiding at all, for times up to a few minutes.

    vrw is the accelerometer noise in m/s/sqrt(s), accel_bias in m/s^2, sigma_v0 in m/s, sigma_tilt0 in rad.
    """
    t = np.asarray(t, dtype=float)
    terms = {'start velocity': sigma_v0 * t,
             'accelerometer bias': accel_bias * t ** 2 / 2.0,
             'accelerometer noise': vrw * np.sqrt(t ** 3 / 3.0),
             'start tilt': g * sigma_tilt0 * t ** 2 / 2.0,
             'gyro bias, through tilt': g * B * t ** 3 / 6.0,
             'gyro noise, through tilt': g * N * np.sqrt(t ** 5 / 20.0)}
    return terms, np.sqrt(sum(x ** 2 for x in terms.values()))


def first_time_over(t, sigma, limit_95, k=1.96):
    """First time at which the 95 percent bound k * sigma passes limit_95, or None if it never does."""
    over = np.nonzero(k * np.asarray(sigma) > limit_95)[0]
    return float(np.asarray(t)[over[0]]) if over.size else None


def plot_terms(t, terms, total, limit_95=None, k=1.96, title=''):
    """Log-log plot of each error term and their total, with the requirement drawn as a 1-sigma line."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name, x in terms.items():
        if np.any(x > 0):
            ax.loglog(t[1:], x[1:], lw=1, label=name)
    ax.loglog(t[1:], total[1:], 'k', lw=2, label='total, 1σ')
    if limit_95 is not None:
        ax.axhline(limit_95 / k, color='tab:red', ls='--', label='requirement, as 1σ')
    ax.set_xlabel('time since the last fix (s)')
    ax.set_ylabel('position error, 1σ (m)')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()

In [ ]:
gyro = gyro_noise()
N, B, K = gyro['N'], gyro['B'], gyro['K']
k1 = K95['1D']
src = gyro['source']
print(f'Gyro z noise ({src}): N = {N:.2e} rad/sqrt(s), B = {B:.2e} rad/s, K = {K:.2e} rad/s/sqrt(s)')
print(f'In datasheet units: N = {np.degrees(N) * 60:.3f} deg/sqrt(h), B = {np.degrees(B) * 3600:.1f} deg/h')

# Example: the mower crosses the 15 m side passage next to the house, where RTK is lost.
V_PASSAGE = 0.3                 # m/s in the passage
T_GAP = 15.0 / V_PASSAGE        # s without a fix: 50 s
SIGMA_Y0 = 0.02                 # m, cross-track error when the fix is lost; Tool 5 gives a better value
SIGMA_PSI0 = np.radians(0.3)    # rad, heading error when the fix is lost; Tool 5 gives a better value
Q_LAT = 0.005 ** 2              # m²/s, lateral wheel slip on grass: an assumption to measure
LIMIT_95 = 0.25                 # m, the part of the 0.325 m wall clearance left for localization

t = np.linspace(0.0, 120.0, 1201)
terms, cross, along = dead_reckoning_drift(t, V_PASSAGE, N, B, K, SIGMA_Y0, SIGMA_PSI0, Q_LAT, scale_err=0.02)
i = np.searchsorted(t, T_GAP)
print(f'After {T_GAP:.0f} s: cross-track {cross[i]:.3f} m (1σ), 95 percent bound {k1 * cross[i]:.3f} m, limit {LIMIT_95} m')
print(f'Along-track after {T_GAP:.0f} s, from a 2 percent odometry scale error: {along[i]:.3f} m (1σ)')
t_over = first_time_over(t, cross, LIMIT_95)
print(f'Dead reckoning stays inside the limit for {t_over:.0f} s' if t_over is not None else 'Inside the limit for all 120 s')
plot_terms(t, terms, cross, LIMIT_95, title='Mower in the side passage: wheels and gyro, no GNSS')

In [ ]:
# Example: the same VN-100 with no wheels and no GNSS, as on a drone or a wearable.
VRW = 0.14e-3 * 9.81          # m/s/sqrt(s), from the datasheet noise density of 0.14 mg/sqrt(Hz)
ACC_BIAS = 0.04e-3 * 9.81     # m/s², from the 0.04 mg in-run bias stability
TILT0 = np.radians(0.5)       # rad, the datasheet static pitch and roll accuracy
t = np.linspace(0.0, 60.0, 6001)
terms, total = inertial_only_drift(t, N, B, VRW, ACC_BIAS, sigma_tilt0=TILT0)
for limit in (0.1, 1.0, 10.0):
    t_over = first_time_over(t, total, limit)
    when = f'after {t_over:.1f} s' if t_over is not None else 'not within 60 s'
    print(f'The 95 percent bound passes {limit:4.1f} m {when}')
plot_terms(t, terms, total, title='VN-100 alone, no aiding')

**In your report.** Show the drift plot for the longest gap in absolute fixes that your mission contains, with the requirement drawn on it, and say which term dominates. If the bias term dominates, a better gyro helps. If the starting heading term dominates, a better heading estimate before the gap helps more, and Tool 5 shows how to get one.

## 4.5 How often you need a fix (Tool 5)

Tool 4 assumes the robot learns nothing new during a gap. A real estimator also learns between gaps: while the robot moves, each position fix reveals not only where it is but also how far its heading and gyro bias have drifted. Tool 5 runs the covariance part of a Kalman filter with three states, the cross-track error $e_y$, the heading error $e_\psi$, and the gyro bias $b$:

$$\dot e_y = v\,e_\psi \qquad \dot e_\psi = b + w_N \qquad \dot b = w_K$$

where $w_N$ and $w_K$ are white noise with the densities $N$ and $K$ from Lab 3. A position fix with 1σ error $\sigma_{fix}$ arrives every $T$ seconds, except inside the outage windows you give it. You do not need to derive the filter. You need it to answer three design questions:
<ul>
    <li> How good is the pose when the robot enters a gap? That is the starting point for Tool 4 </li>
    <li> How large is the error at the end of a known gap, such as a passage, a tunnel, or a canopy? </li>
    <li> How far apart may fixes be before the requirement fails? That becomes a derived requirement on your fix source </li>
</ul>

The model is honest about one thing and simple about others. Honest: heading and bias become observable from position fixes only while the robot moves, so a robot standing still learns nothing about its heading this way. Simple: it assumes straight driving at constant speed and small angles. The turns at the end of each pass help in reality, so treat its answer as slightly pessimistic, and confirm it on the prototype.

Check the tool before you trust it. With no fixes at all it must reproduce Tool 4 exactly, and the first cell after the functions checks that.

In [ ]:
def kalman_update(P, H, r):
    """Joseph-form covariance update for one scalar measurement with variance r."""
    S = (H @ P @ H.T).item() + r
    G = P @ H.T / S
    A = np.eye(P.shape[0]) - G @ H
    return A @ P @ A.T + r * (G @ G.T)


def fix_interval_analysis(v, N, B, K=0.0, q_lat=0.0, fix_sigma=0.02, fix_interval=1.0,
                          sigma_y0=0.02, sigma_psi0=np.radians(2.0), duration=300.0, dt=0.05,
                          outages=(), heading_sigma=None, heading_interval=None):
    """Covariance of a three-state Kalman filter: cross-track error, heading error, and gyro bias.

    The robot drives straight at v m/s and integrates its gyro, with N, B, K in SI units. A position
    fix with 1-sigma error fix_sigma m arrives every fix_interval s, except inside the outage windows
    [(start_s, end_s), ...]. An optional heading measurement, such as a compass or a dual-antenna
    receiver, arrives every heading_interval s with 1-sigma error heading_sigma rad.
    Returns the time and the 1-sigma errors of the three states.
    """
    F = np.array([[1.0, v * dt, v * dt ** 2 / 2.0],
                  [0.0, 1.0, dt],
                  [0.0, 0.0, 1.0]])
    qn, qk = N ** 2, K ** 2
    Q = np.array([
        [q_lat * dt + qn * v ** 2 * dt ** 3 / 3.0 + qk * v ** 2 * dt ** 5 / 20.0,
         qn * v * dt ** 2 / 2.0 + qk * v * dt ** 4 / 8.0,
         qk * v * dt ** 3 / 6.0],
        [0.0, qn * dt + qk * dt ** 3 / 3.0, qk * dt ** 2 / 2.0],
        [0.0, 0.0, qk * dt]])
    Q = Q + np.triu(Q, 1).T                      # fill in the lower triangle
    P = np.diag([sigma_y0 ** 2, sigma_psi0 ** 2, B ** 2])
    H_pos, H_head = np.array([[1.0, 0.0, 0.0]]), np.array([[0.0, 1.0, 0.0]])
    n = int(round(duration / dt))
    every_fix = max(1, int(round(fix_interval / dt)))
    every_head = max(1, int(round(heading_interval / dt))) if heading_interval else 0
    t = np.arange(n + 1) * dt
    sig = np.empty((n + 1, 3))
    sig[0] = np.sqrt(np.diag(P))
    for k in range(1, n + 1):
        P = F @ P @ F.T + Q
        lost = any(start <= t[k] < end for start, end in outages)
        if k % every_fix == 0 and not lost:
            P = kalman_update(P, H_pos, fix_sigma ** 2)
        if every_head and heading_sigma and k % every_head == 0:
            P = kalman_update(P, H_head, heading_sigma ** 2)
        sig[k] = np.sqrt(np.diag(P))
    return t, sig[:, 0], sig[:, 1], sig[:, 2]


def sweep_fix_interval(intervals, limit_95, k=1.96, settle=120.0, **model):
    """Peak 95 percent cross-track error just before each fix, once the filter has settled."""
    rows = []
    for T in intervals:
        t, sy, _, _ = fix_interval_analysis(fix_interval=T, duration=settle + 5 * T, **model)
        peak = k * sy[t >= settle].max()
        verdict = 'meets' if peak <= limit_95 else 'fails'
        print(f'fix every {T:5.1f} s: peak 95 percent error {peak:.3f} m, limit {limit_95:.3f} m: {verdict}')
        rows.append((T, peak, peak <= limit_95))
    return rows

In [ ]:
# Check the tool on a case whose answer you know: with no fixes at all, the filter's
# cross-track error must grow exactly as Tool 4 predicts.
t5, sy5, _, _ = fix_interval_analysis(v=0.5, N=N, B=B, K=K, q_lat=Q_LAT, sigma_y0=0.02,
                                      sigma_psi0=np.radians(1.0), duration=100.0, outages=[(0.0, 1e9)])
_, cross4, _ = dead_reckoning_drift(t5, 0.5, N, B, K, 0.02, np.radians(1.0), Q_LAT)
print(f'Largest difference between Tool 5 and Tool 4 over 100 s: {np.max(np.abs(sy5 - cross4)):.1e} m')

In [ ]:
# Example: the mower drives at 0.3 m/s on RTK fixed solutions at 5 Hz, about 0.02 m each (Lab 2),
# then enters the side passage, where there is no fix for 50 s.
OUTAGE = (200.0, 200.0 + T_GAP)
t, sy, spsi, sb = fix_interval_analysis(v=V_PASSAGE, N=N, B=B, K=K, q_lat=Q_LAT, fix_sigma=0.02,
                                        fix_interval=0.2, sigma_psi0=np.radians(2.0),
                                        duration=300.0, outages=[OUTAGE])
i0 = np.searchsorted(t, OUTAGE[0]) - 1
i1 = np.searchsorted(t, OUTAGE[1]) - 1
print(f'Entering the passage: cross-track {1000 * sy[i0]:.1f} mm, heading {np.degrees(spsi[i0]):.2f} deg, '
      f'gyro bias {np.degrees(sb[i0]) * 3600:.1f} deg/h, all 1σ')
print(f'Leaving the passage: cross-track {sy[i1]:.3f} m (1σ), 95 percent bound {k1 * sy[i1]:.3f} m, '
      f'limit {LIMIT_95} m')

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(t, k1 * sy, label='cross-track error, 95 percent bound')
ax.axvspan(*OUTAGE, color='gray', alpha=0.2, label='no fix in the passage')
ax.axhline(LIMIT_95, color='tab:red', ls='--', label='limit in the passage')
ax.set_xlabel('time (s)')
ax.set_ylabel('cross-track error (m)')
ax.grid(alpha=0.3, which='both')
ax.legend()
plt.show()

In [ ]:
# How far apart may fixes be on the open lawn? R4 allows 0.035 m at 1σ, so 0.069 m at 95 percent.
lawn = dict(v=0.5, N=N, B=B, K=K, q_lat=Q_LAT, fix_sigma=0.02, sigma_psi0=np.radians(2.0))
rows = sweep_fix_interval([0.2, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0], limit_95=k1 * 0.035, **lawn)
t, sy, _, _ = fix_interval_analysis(fix_interval=0.2, duration=300.0, **lawn)
SIGMA_OPEN = float(sy[t >= 280.0].max())
print(f'Open lawn with 5 Hz fixes: {1000 * SIGMA_OPEN:.1f} mm (1σ). This is the localization entry in Tool 6.')

**In your report.** Give the pose quality when the robot enters its worst gap, the error at the end of that gap, and the longest fix interval that still meets your accuracy requirement. State that interval as a derived requirement on your absolute positioning source. If your fixes come from something other than GNSS, such as markers or scan matching, the same tool applies with their spacing and noise.

## 4.6 The error budget (Tool 6)

An error budget adds up every independent error source between the truth and the quantity the requirement measures, and compares the total with the requirement:

$$\sigma_{total} = \sqrt{\sigma_1^2 + \sigma_2^2 + \dots} \qquad e_{95} = k\,\sigma_{total}$$

with $k = 1.96$ for one axis, such as cross-track error, and $k = 2.45$ for a horizontal radius. Include more than the sensor: localization, the map, path tracking, calibration (lever arms, extrinsics), time stamps (speed times time error), and any bias from the environment. The budget also runs backwards. Fix every other entry, and `allowed_sigma` returns the most that one entry may contribute. That number is a derived requirement, and it is how the worked example gets R4.

Label every entry as precision or accuracy, with its source. The Lab 2 deduction applies: treating a precision number as an accuracy number costs points.

In [ ]:
def error_budget(contributors, requirement_95, dims='1D', title='Error budget'):
    """Root sum square of independent 1-sigma contributors, compared with a 95 percent requirement.

    contributors: list of (name, sigma_m, source). dims is '1D' for one axis or '2D' for a horizontal radius.
    """
    k = K95[dims]
    names = [c[0] for c in contributors]
    sig = np.array([c[1] for c in contributors], dtype=float)
    total = float(np.sqrt(np.sum(sig ** 2)))
    share = sig ** 2 / total ** 2
    width = max(len(n) for n in names)
    print('Contributor'.ljust(width) + '   1σ (m)  share  source')
    for (name, s, src), f in zip(contributors, share):
        print(f'{name:{width}s}   {s:6.3f}  {100 * f:4.0f}%  {src}')
    bound = k * total
    verdict = 'meets' if bound <= requirement_95 else 'FAILS'
    print(f'Total {total:.3f} m (1σ), {bound:.3f} m at 95 percent, against {requirement_95:.3f} m required: '
          f'{verdict}, margin {requirement_95 - bound:+.3f} m')
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(names) + 1.2))
    order = np.argsort(share)
    ax.barh([names[i] for i in order], 100 * share[order])
    ax.set_xlabel('share of the total variance (percent)')
    ax.set_title(f'{title}: {bound:.3f} m at 95 percent, {requirement_95:.3f} m required')
    ax.grid(alpha=0.3, axis='x')
    plt.show()
    return total, bound


def allowed_sigma(requirement_95, other_sigmas, dims='1D'):
    """Largest 1-sigma value one contributor may have once all the others are fixed."""
    room = (requirement_95 / K95[dims]) ** 2 - float(np.sum(np.asarray(other_sigmas, dtype=float) ** 2))
    return float(np.sqrt(room)) if room > 0 else 0.0

In [ ]:
# Example: the mower on boundary passes. R1: at most 0.10 m outside the boundary, 95 percent.
R1_LIMIT = 0.10
others = [('boundary map, RTK teach-in walk', 0.020, 'Lab 2 fixed precision, pole held by hand'),
          ('path tracking, pure pursuit on grass', 0.030, 'assumption, to test on the prototype'),
          ('antenna lever arm, tilt-corrected', 0.005, '1 deg tilt error x 0.30 m antenna height'),
          ('time stamps, 10 ms at 0.5 m/s', 0.005, 'GNSS time, with the latency compensated')]
sigma_loc = allowed_sigma(R1_LIMIT, [s for _, s, _ in others])
print(f'Localization may contribute at most {sigma_loc:.4f} m (1σ). This becomes derived requirement R4.')
print()
error_budget([('localization, RTK fixed', SIGMA_OPEN, 'Tool 5, 5 Hz fixes')] + others,
             R1_LIMIT, title='Boundary pass, open sky')
print()
error_budget([('localization, RTK float', 0.40, 'Lab 2 float precision')] + others,
             R1_LIMIT, title='Boundary pass under the tree, float')

In [ ]:
# The design change: under the tree, ignore float solutions for boundary control and dead-reckon
# from the last fixed solution. How far may a boundary pass run under the canopy and still meet R4?
t, sy, _, _ = fix_interval_analysis(v=V_PASSAGE, N=N, B=B, K=K, q_lat=Q_LAT, fix_sigma=0.02,
                                    fix_interval=0.2, sigma_psi0=np.radians(2.0),
                                    duration=300.0, outages=[(200.0, 300.0)])
over = np.nonzero((t >= 200.0) & (sy > sigma_loc))[0]
t_max = t[over[0]] - 200.0 if over.size else 100.0
L_MAX = V_PASSAGE * t_max
print(f'At {V_PASSAGE} m/s, dead reckoning meets R4 for {t_max:.1f} s after the last fixed solution.')
print(f'A boundary pass may run at most {L_MAX:.1f} m under the canopy; derived requirement R11 rounds it down to {np.floor(L_MAX):.0f} m.')

**In your report.** Show the budget for your tightest accuracy requirement, in the worst place in the mission, and say what you changed when it failed. The worked example's budget fails under a tree, and the fix is a change to the planner, not a new sensor.

## 4.7 Mapping, planning, and control

Say what the map contains, how it is built, how big it is, and how it is kept current when the world changes. Say which planner and controller you use, and estimate the controller's tracking error, which is an entry in your error budget. Give the final approach or docking its own paragraph, because it is often the tightest requirement in the whole system.

### Pass 1, on your own

1. What is your largest error source, and what would it cost to halve it?
2. Where in the mission is the absolute fix weakest, and for how long?
3. What happens when your map is wrong?

# Part 5: System Integration

## 5.1 Frames and calibration

Draw the TF tree, `map → odom → base_link →` every sensor frame, and give each static transform with the source of its numbers: CAD, a tape measure, or a calibration routine. Then give a calibration plan. Example rows:

| What | How | How often | Acceptance check |
|---|---|---|---|
| Gyro bias | Stand still for the averaging time from your Allan curve | Every start | Change below B |
| Wheel odometry scale | Drive a measured 10 m line | Weekly | Error below 0.5 percent |
| Antenna lever arm | Tape measure from `base_link` | Once | ±5 mm |
| Camera intrinsics and extrinsics | Calibration target | After any knock | Reprojection error below 0.5 px |

## 5.2 Time

Every measurement needs a time stamp from one clock, and you need to know how wrong that stamp can be. A time error $\delta t$ at speed $v$ is a position error $v\,\delta t$: 10 ms at 2 m/s is 2 cm. Lab 3 showed that a simple driver stamps the time a line arrives, not the time the sensor sampled it. Say how your computers share one clock (chrony or PTP), where the GNSS time and its PPS pulse go, and which latencies you measure and compensate.

## 5.3 Data rates and storage (Tool 7)

Add up the bandwidth of every topic, then compare the totals with each link they cross and with the storage your logging needs. Cameras and 3D lidars dominate every budget, so check them first. Message sizes in the example are estimates; measure your own with `ros2 topic bw` on real topics when you can.

In [ ]:
def data_budget(topics, link_mbps=None, log_hours=None):
    """Bandwidth of each topic, and the totals over a network link and into the log.

    topics: list of (name, bytes_per_message, rate_hz, over_link, logged). link_mbps is in Mbit/s.
    """
    def yes(flag):
        return 'yes' if flag else 'no'

    width = max(len(x[0]) for x in topics)
    print('Topic'.ljust(width) + '     MB/s  link  log')
    link = log = 0.0
    for name, size, rate, over_link, logged in topics:
        mbs = size * rate / 1e6
        link += mbs if over_link else 0.0
        log += mbs if logged else 0.0
        print(f'{name:{width}s}  {mbs:7.3f}  {yes(over_link):>4s}  {yes(logged):>3s}')
    note = f' of {link_mbps} Mbit/s, {100 * 8 * link / link_mbps:.0f} percent' if link_mbps else ''
    print(f'Over the link: {link:.3f} MB/s = {8 * link:.2f} Mbit/s{note}')
    if log_hours:
        gb_per_h = log * 3600 / 1000
        print(f'Into the log: {log:.3f} MB/s, {gb_per_h:.1f} GB per hour, {gb_per_h * log_hours:.1f} GB for {log_hours} h')
    return link, log

In [ ]:
# Example: the mower. Message sizes are estimates; measure yours with ros2 topic bw.
topics = [
    # name, bytes per message, rate in Hz, over Wi-Fi to the phone app, logged
    ('/oak/stereo/depth, 640 x 480 x 2 bytes', 640 * 480 * 2, 15, False, True),
    ('/oak/rgb/image_raw/compressed', 60_000, 2, False, True),
    ('/obstacles', 400, 15, False, True),
    ('/ultrasonic/range, three sensors', 3 * 60, 10, False, True),
    ('/gnss/fix, NavSatFix', 130, 5, False, True),
    ('/imu, sensor_msgs/Imu', 320, 100, False, True),
    ('/wheel/odometry', 720, 50, False, True),
    ('/odometry/filtered, two EKFs', 2 * 720, 30, False, True),
    ('/tf', 300, 50, False, True),
    ('/cmd_vel', 70, 20, False, True),
    ('/status to the phone app', 2_000, 1, True, False),
]
print('Everything logged, depth at 15 Hz:')
data_budget(topics, link_mbps=20, log_hours=2)
print()
print('Depth logged through a 1 Hz throttled copy instead:')
throttled = [(n, size, r, o, g and not n.startswith('/oak/stereo')) for n, size, r, o, g in topics]
throttled.append(('/oak/stereo/depth_throttled, 1 Hz', 640 * 480 * 2, 1, False, True))
data_budget(throttled, link_mbps=20, log_hours=2)

## 5.4 Compute

Give a short table: each node, where it runs, its expected CPU share, and how you know, whether you measured it with `top` on a prototype, took it from a published benchmark, or guessed and will test. Keep about 30 percent headroom on every processor. Heavy perception on a camera's own processor, as on an OAK-D, saves the main computer.

## 5.5 Power and runtime (Tool 8)

Follow Lecture 06. List every load with its average and peak power, divide the electronics by the converter efficiency, apply the usable fraction of the battery, aging, and cold, and keep a reserve to get back to the charger. Then check the peak current against the pack's continuous C-rate limit. Design for the old pack on the worst day, not the new pack on a mild one.

In [ ]:
def power_budget(loads, battery, converter_eff=0.90, required_min=None, reserve=0.15, design_case='aged pack'):
    """Average and peak power, and runtime on a new pack, an aged pack, and an aged pack in the cold.

    loads: list of (name, average_W, peak_W, on_converter). Loads on the DC-DC converter are
           divided by its efficiency, as in Lecture 06.
    battery: dict with volts, amp_hours, usable (the fraction of the label you may use), aged, cold,
             and max_c_rate (the pack's continuous discharge limit).
    reserve: the fraction of usable energy kept to get back to the charger; it does not count as runtime.
    """
    width = max(len(x[0]) for x in loads)
    print('Load'.ljust(width) + '   avg W  peak W')
    avg = peak = 0.0
    for name, a, p, on_converter in loads:
        f = 1.0 / converter_eff if on_converter else 1.0
        avg += a * f
        peak += p * f
        print(f'{name:{width}s}  {a * f:6.1f}  {p * f:6.1f}')
    wh = battery['volts'] * battery['amp_hours']
    usable = wh * battery['usable']
    cases = {'new pack': usable,
             'aged pack': usable * battery['aged'],
             'aged pack, cold': usable * battery['aged'] * battery['cold']}
    print(f'Total {avg:.1f} W average and {peak:.1f} W peak, from a {wh:.0f} Wh pack')
    minutes = {}
    for case, e in cases.items():
        minutes[case] = 60.0 * e * (1.0 - reserve) / avg
        print(f'   {case:16s} {e:6.1f} Wh usable  {minutes[case]:5.0f} min before the reserve')
    amps = peak / battery['volts']
    c_rate = amps / battery['amp_hours']
    limit = battery['max_c_rate']
    c_verdict = 'within' if c_rate <= limit else 'OVER'
    print(f'Peak current {amps:.1f} A, which is {c_rate:.2f}C: {c_verdict} the {limit}C limit')
    if required_min is not None:
        got = minutes[design_case]
        verdict = 'meets' if got >= required_min else 'FAILS'
        print(f'Requirement: {required_min} min on the {design_case}: {verdict} with {got:.0f} min')
    return minutes

In [ ]:
# Example: the mower. The electronics run from a DC-DC converter; the motors run from the pack.
# Motor, computer, and microcontroller powers are estimates to measure; sensor values are from datasheets.
loads = [
    # name, average W, peak W, on the converter
    ('blade motor', 30.0, 150.0, False),
    ('drive motors, both', 16.0, 80.0, False),
    ('Raspberry Pi 5', 6.0, 12.0, True),
    ('OAK-D Lite', 3.0, 5.0, True),
    ('RTK GNSS receiver', 0.4, 0.5, True),
    ('IMU', 0.2, 0.2, True),
    ('ultrasonic sensors, three', 0.1, 0.1, True),
    ('safety microcontroller and switches', 0.3, 0.5, True),
]
pack = {'volts': 18.0, 'amp_hours': 5.0, 'usable': 0.90, 'aged': 0.80, 'cold': 0.85, 'max_c_rate': 2.0}
for ah in (5.0, 7.5):
    print(f'Pack: 18 V, {ah} Ah')
    power_budget(loads, dict(pack, amp_hours=ah), required_min=60)
    print()

## 5.6 Parts list, cost, and mass (Tool 9)

List every part of the prototype you could build in the final project: what it is, the model, the quantity, the unit price, mass, power, where you would buy it, and whether the course owns it. Course-owned parts count toward mass and power but not toward cost. Masses and prices you have not checked are estimates, so say so in the source column, and weigh the parts on a kitchen scale once you have them. If your client gave a unit cost limit, price the product version as well, as the worked example does.

In [ ]:
def parts_list(items, budget_usd=None):
    """Out-of-pocket cost, added mass, and added power of a prototype.

    items: list of dicts with part, model, qty, unit_usd (None when unknown), mass_kg, power_w, source,
           and owned (True for course hardware, which costs you nothing).
    """
    wp = max(len(i['part']) for i in items)
    wm = max(len(i['model']) for i in items)
    print('Part'.ljust(wp) + '  ' + 'Model'.ljust(wm) + '  qty    unit $  owned  mass kg  power W  source')
    cost = mass = power = 0.0
    for i in items:
        u = i['unit_usd']
        unit = '      n/a' if u is None else f'{u:9.2f}'
        own = 'yes' if i['owned'] else 'no'
        if not i['owned'] and u is not None:
            cost += i['qty'] * u
        mass += i['qty'] * i['mass_kg']
        power += i['qty'] * i['power_w']
        part, model, qty = i['part'], i['model'], i['qty']
        m, p, s = i['mass_kg'], i['power_w'], i['source']
        print(f'{part:{wp}s}  {model:{wm}s}  {qty:3d}  {unit}  {own:>5s}  {m:7.3f}  {p:7.2f}  {s}')
    print(f'Out of pocket ${cost:.2f}, added mass {mass:.2f} kg, added power {power:.1f} W')
    if budget_usd is not None:
        verdict = 'meets' if cost <= budget_usd else 'OVER'
        print(f'Prototype budget ${budget_usd}: {verdict}')
    return cost, mass, power

In [ ]:
# Example: the mower's final project prototype, a TurtleBot 4 on a paved courtyard with a taped boundary.
# Prices are list prices at the time the example was written; masses marked as estimates must be weighed.
prototype = [
    {'part': 'Platform', 'model': 'TurtleBot 4', 'qty': 1, 'unit_usd': None,
     'mass_kg': 0.0, 'power_w': 0.0, 'source': 'course', 'owned': True},
    {'part': 'RTK GNSS receiver and antenna', 'model': 'SparkFun RTK mosaic-X5 kit', 'qty': 1, 'unit_usd': 1249.95,
     'mass_kg': 0.35, 'power_w': 2.5, 'source': 'course, Lab 2; mass and power estimated', 'owned': True},
    {'part': 'IMU', 'model': 'VectorNav VN-100 Rugged', 'qty': 1, 'unit_usd': None,
     'mass_kg': 0.015, 'power_w': 0.2, 'source': 'course, Lab 3; datasheet', 'owned': True},
    {'part': 'Ultrasonic sensor', 'model': 'DFRobot A02YYUW (SEN0311)', 'qty': 3, 'unit_usd': 15.90,
     'mass_kg': 0.02, 'power_w': 0.04, 'source': 'dfrobot.com; mass estimated', 'owned': False},
    {'part': 'Ultrasonic interface', 'model': 'Raspberry Pi Pico 2', 'qty': 1, 'unit_usd': 5.00,
     'mass_kg': 0.004, 'power_w': 0.1, 'source': 'Raspberry Pi resellers', 'owned': False},
    {'part': 'Power for the receiver', 'model': 'USB-C power bank, 10,000 mAh', 'qty': 1, 'unit_usd': 20.00,
     'mass_kg': 0.20, 'power_w': 0.0, 'source': 'estimate', 'owned': False},
    {'part': 'Sensor mounts', 'model': '3D-printed PLA', 'qty': 1, 'unit_usd': 5.00,
     'mass_kg': 0.08, 'power_w': 0.0, 'source': 'estimate, filament only', 'owned': False},
    {'part': 'Boundary marking', 'model': 'painter tape, 2 rolls', 'qty': 1, 'unit_usd': 10.00,
     'mass_kg': 0.0, 'power_w': 0.0, 'source': 'estimate', 'owned': False},
]
parts_list(prototype, budget_usd=150)

## 5.7 Verification plan and traceability (Tool 10)

For each key requirement, plan how you would show that it is met: the method (test, analysis, inspection, or demonstration), the test itself, the pass criterion, the equipment, and whether you could run it in the final project with course hardware. Example row:

| Requirement | Method | Test | Pass criterion | Equipment | In the final project? |
|---|---|---|---|---|---|
| R1 boundary | Test | 20 laps of a taped boundary on pavement, with chalk at the body edge drawing the path | 95th percentile of the overshoot at most 0.10 m | TurtleBot 4, RTK receiver, tape measure | Yes |

Then check traceability. Every requirement needs at least one design element that meets it and one test that checks it. Every design element needs a requirement, or it is cost with no reason. Tool 10 finds the gaps; in the example below it finds two, on purpose.

In [ ]:
def traceability(requirements, design, verification):
    """Every requirement met by something and checked by something, and every part there for a reason.

    requirements: {id: text}. design: {element: [requirement ids it serves]}. verification: {id: test}.
    """
    def cell(flag):
        return 'x' if flag else '.'

    ids = list(requirements)
    width = max(len(e) for e in list(design) + ['verification'])
    print(' ' * (width + 1) + ''.join(f'{r:>5s}' for r in ids))
    for element, served in design.items():
        print(f'{element:{width}s} ' + ''.join(f'{cell(r in served):>5s}' for r in ids))
    label = 'verification'
    print(f'{label:{width}s} ' + ''.join(f'{cell(r in verification):>5s}' for r in ids))
    print()
    problems = []
    for r in ids:
        if not any(r in served for served in design.values()):
            problems.append(f'{r} has no design element: nothing in the design meets it')
        if r not in verification:
            problems.append(f'{r} has no verification: no one will know whether it is met')
    for element, served in design.items():
        if not served:
            problems.append(f'{element} serves no requirement: remove it, or find the requirement you forgot')
        for r in served:
            if r not in requirements:
                problems.append(f'{element} lists {r}, which is not a requirement ID')
    for p in problems:
        print(p)
    print('Traceability: no problems found' if not problems else f'Traceability: {len(problems)} problems to fix')
    return problems

In [ ]:
# Example: the mower, with two gaps left in on purpose so you can see what the check reports.
requirements = {
    'R1': 'Body at most 0.10 m outside the boundary, 95 percent',
    'R2': 'Mows at least 95 percent of the lawn each session',
    'R3': 'No buried boundary wire',
    'R4': 'Cross-track localization error at most 0.035 m (1σ) on boundary passes',
    'R5': 'Stops at least 0.15 m short of a 0.3 m tall object',
    'R6': 'Detects a 0.3 m tall object at 0.6 m or more',
    'R7': 'At least 60 min of mowing per charge on an aged pack, with a 15 percent reserve',
    'R8': 'Crosses the 15 m side passage without GNSS and without touching the walls',
    'R9': 'Blade stops within 1 s of a lift, or of a tilt beyond 30 degrees',
    'R10': 'Sensing and computing parts cost at most $1,000 per mower',
    'R11': f'Boundary passes run at most {np.floor(L_MAX):.0f} m past the last RTK fixed solution',
}
design = {
    'RTK GNSS receiver': ['R1', 'R3', 'R4'],
    'IMU': ['R4', 'R8', 'R9'],
    'wheel encoders': ['R4', 'R8'],
    'depth camera': ['R5', 'R6'],
    'ultrasonic array': ['R5', 'R6'],
    'bumper, lift, and tilt switches': ['R5', 'R9'],
    'EKF, robot_localization': ['R1', 'R4', 'R8'],
    'coverage planner': ['R2', 'R11'],
    'pure pursuit controller': ['R1', 'R8'],
    'safety microcontroller': ['R9'],
    '18 V, 7.5 Ah pack and dock': ['R7'],
    'product parts list': ['R10'],
    'magnetometer': [],
}
verification = {
    'R1': 'T1 boundary laps', 'R2': 'T2 coverage map', 'R3': 'inspection', 'R4': 'Tool 6, then T1',
    'R5': 'T3 foam block trials', 'R6': 'T3 foam block trials', 'R7': 'T4 runtime with a dummy blade load',
    'R8': 'T5 passage mock-up with the antenna covered', 'R9': 'T6 lift and tilt trials',
    'R10': 'analysis with Tool 9',
}
problems = traceability(requirements, design, verification)

**In your report.** Show the traceability matrix and say what you fixed. The two problems in the example are typical: a requirement added late, R11, with no test yet, and a sensor carried over from an early idea, the magnetometer, that no requirement needs. The worked example fixes both.

### Pass 1, on your own

1. Which clock stamps each measurement?
2. Which link, bus, or battery is closest to its limit?
3. What is the worst-case runtime: old pack, cold day, everything on?
4. Which three tests would you run first on a prototype, with course hardware?

# Part 6: Failure Analysis

## 6.1 Find the failure modes

Walk your pipeline figure block by block, including power, computing, the network, the map, people, and the environment, and ask the same questions of every block:

| Guide word | Ask |
|---|---|
| None | What if it gives no output at all? |
| Stuck | What if it keeps repeating the last value? |
| Biased | What if it is wrong but plausible? |
| Noisy | What if the noise is ten times the datasheet value? |
| Late | What if it arrives 200 ms late, or out of order? |
| Intermittent | What if it drops out for 2 s every minute? |
| Wrong frame or units | What if degrees arrive as radians, or gauss as tesla? |

The biased case is the dangerous one, because nothing looks broken. The labs have already shown you several: an RTK solution read as fixed when it was float, time stamps that are arrival times, a magnetometer next to a motor, and voltage sag that resets a sensor.

## 6.2 Rank them (Tool 11)

Score each failure mode from 1 to 10 for severity S, occurrence O, and detection D, where a high D means hard to detect. The risk priority number is RPN = S × O × D.

| Score | Severity S | Occurrence O | Detection D |
|---|---|---|---|
| 1 | No effect | Practically never | Always detected before any effect |
| 3 | A minor delay | Rare, once a season | Very likely detected |
| 5 | The mission fails, with no damage | Occasional, once a month | Detected about half the time |
| 7 | Property damage, or a stranded robot | Frequent, once a week | Rarely detected |
| 9 | Injury possible, with warning | Almost every mission | Detected only after harm |
| 10 | Injury possible, without warning | Every mission | Cannot be detected |

Two rules matter more than the ranking. Every failure with severity 9 or 10 gets an action, whatever its RPN, because the product of three ordinal scores can hide a rare and catastrophic failure. And a detection must not rely on the part that failed: a crashed computer cannot report its own crash.

In [ ]:
def fmea(rows, top=10):
    """Risk priority numbers before and after each action, ranked, with every severity 9 or 10 row flagged.

    rows: list of dicts with mode, S, O, D, detection, action, and optionally S2, O2, D2 after the action.
    """
    for r in rows:
        r['RPN'] = r['S'] * r['O'] * r['D']
        r['RPN2'] = r['S2'] * r['O2'] * r['D2'] if all(key in r for key in ('S2', 'O2', 'D2')) else None
    ranked = sorted(rows, key=lambda r: (-r['RPN'], -r['S']))
    width = max(len(r['mode']) for r in ranked)
    print('Failure mode'.ljust(width) + '    S  O  D  RPN  after')
    for r in ranked:
        mode, s, o, d, rpn, rpn2 = r['mode'], r['S'], r['O'], r['D'], r['RPN'], r['RPN2']
        after = '' if rpn2 is None else f'{rpn2:5d}'
        flag = '   severity 9 or 10' if s >= 9 else ''
        print(f'{mode:{width}s}  {s:3d}{o:3d}{d:3d}  {rpn:3d}  {after}{flag}')
    print()
    print('Severity 9 or 10, whatever the RPN:')
    for r in ranked:
        if r['S'] >= 9:
            mode, det, act = r['mode'], r['detection'], r['action']
            print(f'   {mode}: detected by {det}; action: {act}')

    shown = ranked[:top]
    y = np.arange(len(shown))
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(shown) + 1.2))
    ax.barh(y, [r['RPN'] for r in shown], color='lightgray', label='before')
    ax.barh(y, [r['RPN2'] or 0 for r in shown], height=0.5, color='tab:blue', label='after the action')
    ax.set_yticks(y)
    ax.set_yticklabels([r['mode'] for r in shown], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('risk priority number, S x O x D')
    ax.set_title(f'The top {len(shown)} failure modes')
    ax.legend()
    fig.tight_layout()
    plt.show()
    return ranked

In [ ]:
# Example: the mower. Scores use the scale above; the worked example explains each row.
failures = [
    {'mode': 'RTK drops to float under the tree', 'S': 6, 'O': 7, 'D': 3,
     'detection': 'the GGA quality field, parsed', 'action': 'stop boundary passes at the R11 distance past the last fix',
     'S2': 6, 'O2': 3, 'D2': 2},
    {'mode': 'NTRIP corrections lost, home Wi-Fi down', 'S': 7, 'O': 4, 'D': 2,
     'detection': 'correction age over 10 s', 'action': 'stop boundary passes, mow the interior, then dock',
     'S2': 3, 'O2': 4, 'D2': 2},
    {'mode': 'Wrong RTK fix near the house wall', 'S': 7, 'O': 2, 'D': 8,
     'detection': 'EKF innovation test against wheels and gyro', 'action': 'reject and count; stop after three in a row',
     'S2': 7, 'O2': 2, 'D2': 3},
    {'mode': 'Wheel slip on wet grass', 'S': 5, 'O': 6, 'D': 5,
     'detection': 'wheel speed against gyro rate and RTK speed', 'action': 'inflate the odometry noise and slow down',
     'S2': 5, 'O2': 6, 'D2': 2},
    {'mode': 'Camera blinded by low sun or a wet lens', 'S': 9, 'O': 4, 'D': 6,
     'detection': 'fraction of valid depth pixels', 'action': 'ultrasonic array as a second, different sensor; slow to 0.2 m/s',
     'S2': 9, 'O2': 2, 'D2': 3},
    {'mode': 'Pet approaches from the side, in the blind zone', 'S': 9, 'O': 3, 'D': 6,
     'detection': 'side ultrasonic and the bumper', 'action': 'bumper all around, cutting blade power in hardware',
     'S2': 9, 'O2': 2, 'D2': 2},
    {'mode': 'Mower lifted with the blade spinning', 'S': 10, 'O': 3, 'D': 2,
     'detection': 'lift switch, and tilt from the IMU', 'action': 'hardware interlock on blade power, tested at every start',
     'S2': 10, 'O2': 1, 'D2': 1},
    {'mode': 'Main computer hangs, last command repeats', 'S': 8, 'O': 3, 'D': 5,
     'detection': 'motor controller watchdog, 0.5 s', 'action': 'motors and blade stop without the computer',
     'S2': 8, 'O2': 3, 'D2': 1},
    {'mode': 'Voltage sag at blade start resets the receiver', 'S': 6, 'O': 3, 'D': 4,
     'detection': 'receiver restart message, rail voltage log', 'action': 'separate rails and a blade soft start',
     'S2': 6, 'O2': 1, 'D2': 2},
    {'mode': 'Magnetometer heading bent by the motors', 'S': 5, 'O': 8, 'D': 7,
     'detection': 'nothing reliable', 'action': 'remove the magnetometer; heading from RTK motion and the gyro',
     'S2': 5, 'O2': 1, 'D2': 1},
    {'mode': 'GNSS stamped on arrival, 80 ms late', 'S': 4, 'O': 8, 'D': 7,
     'detection': 'RTK track against odometry in turns', 'action': 'use GNSS time and compensate the measured latency',
     'S2': 4, 'O2': 2, 'D2': 3},
    {'mode': 'Boundary map wrong after the teach-in walk', 'S': 6, 'O': 5, 'D': 6,
     'detection': 'a second walk, and the map shown on the phone', 'action': 'owner confirms the map before the first run',
     'S2': 6, 'O2': 3, 'D2': 3},
    {'mode': 'Heavy rain', 'S': 6, 'O': 5, 'D': 2,
     'detection': 'rain sensor', 'action': 'stop and return to the dock',
     'S2': 3, 'O2': 5, 'D2': 2},
    {'mode': 'Misses the dock contacts', 'S': 4, 'O': 4, 'D': 3,
     'detection': 'no charge current after docking', 'action': 'back out and retry, aligned on the dock marker',
     'S2': 4, 'O2': 2, 'D2': 1},
    {'mode': 'Stuck in the side passage longer than planned', 'S': 6, 'O': 3, 'D': 3,
     'detection': 'position bound over 0.25 m', 'action': 'stop, blade off, alert the owner',
     'S2': 3, 'O2': 3, 'D2': 2},
]
ranked = fmea(failures)

**In your report.** Give the ranked table, every severity 9 or 10 row with its detection and action, and the top five after your actions. Mark the rows that an AI review suggested, as the AI use log requires.

## 6.3 Detection, recovery, and what is left

For the top five failures and every severity 9 or 10 failure, give the detection signal and its threshold, the response, the degraded mode it puts the system in, and how the system leaves that mode. These are the modes of Part 2, now with numbers. Then state the residual risk: what is left after your actions, and which test you would run first to find out whether you are wrong.

### Pass 1, on your own

1. Which single failure could hurt someone? How is it detected without relying on the part that failed?
2. Which of your safety functions still work when the main computer has crashed?
3. Which two sensors share a weakness, so that one event takes out both?

# Part 7: The report and the five-slide talk

## The report

One PDF, 8 to 12 pages without the references and appendices, in 11 pt type or larger, with every figure numbered, captioned, and readable when printed. Use the six section headings from the syllabus, in the order of `EECE5554_SDE_report_template.md`:

1. Problem Definition
2. System Architecture
3. Sensor Selection
4. Localization, Mapping & Navigation
5. System Integration
6. Failure Analysis

Start with a summary of at most 150 words: the problem, the design in two sentences, the headline numbers, and the biggest open risk. Follow it with a short table of your design loops: what failed, what you changed, and where in the report. End with the references and three appendices:
<ul>
    <li> <b>Appendix A, Tools used:</b> every tool, with its version, what you used it for, and where in the report the result appears </li>
    <li> <b>Appendix B, AI use log</b> and your signed statement, as described in <i>Using AI tools</i> below </li>
    <li> <b>Appendix C, Calculations:</b> your notebook exported to PDF or HTML, or a link to it in your repository </li>
</ul>

Three items appear in both the report and the talk:

| Item | In the report | In the talk |
|---|---|---|
| The pipeline | Section 2 figure, sensors to actuators, with rates | Slide 2 |
| Prototype components | Section 3 parts list, from Tool 9 | Slide 3 |
| Tools used | Appendix A | One line on Slide 5 |

## The talk: five slides, five minutes

This talk is a warm up for your final project presentation: the same discipline, a smaller scope. A timer runs. Aim to finish by 5:00; the talk is stopped at 5:30. About three minutes of questions follow, which you answer without notes.

| Slide | Its title states a claim, for example | It must show |
|---|---|---|
| 1. Problem | A wire-free mower that never leaves the lawn | Your name and scenario in the header, the setting, the three to five requirements that drive the design with their numbers, and what success means |
| 2. System | RTK, wheels, and a gyro feed one estimator; the safety chain needs no software | The pipeline figure with rates, the main ROS 2 nodes, and where the safe stop lives |
| 3. Sensors and prototype | A depth camera finds pets in grass; ultrasonics and a bumper back it up | The trade study result in numbers and the runner-up, placement and blind zones, and the parts list summary: out-of-pocket cost, mass, and power |
| 4. Localization and integration | The boundary budget closes in the open, and the planner handles the tree | One figure that proves your accuracy requirement is met, or shows where it is not, and the runtime result |
| 5. Risks and next steps | Three failures we designed for, and the test we would run first | The top three failure modes with detection and response, the open risk, the first prototype test for the final project, and one line listing the tools you used |

Rules for the slides:
<ul>
    <li> Exactly five slides: no title slide and no backup slides </li>
    <li> Each title is a sentence that states the slide's claim, not a one-word topic </li>
    <li> A figure or a table on every slide, and a unit on every number </li>
    <li> Text at 18 pt or larger, and no paragraphs </li>
    <li> About one minute per slide. Rehearse twice with a timer </li>
    <li> Submit the slides as one PDF; you present from that file </li>
</ul>

Expect questions like these: Where does this number come from? What happens when this sensor fails? Why not the runner-up? What would make your design wrong? What would you test first?

# Using AI tools

The point of this exercise is that you can do system design yourself, with confidence, when no one hands you the answer. AI tools can make you faster and can catch your mistakes, but only if the thinking is yours first. The rule for this exercise: **you are the designer. An AI tool may be your tutor or your reviewer, never your author.**

## The two-pass rule

For every Part:

1. **Pass 1, alone.** Answer that Part's Pass 1 questions in `design_journal.md`, sketch, and run the tools with your own numbers. Then commit the journal. The commit time is your evidence that the thinking came first.
2. **Pass 2, with review.** Only now may you ask an AI tool to review what you wrote. Record every use in your AI use log, including what you changed and why.

Work that appears only in Pass 2, with no Pass 1 behind it in your journal, earns no process points.

## What is allowed

| Zone | What | Examples |
|---|---|---|
| Green: allowed, log it | Learning a concept; finding a datasheet, which you then read yourself; explaining an error in your code; grammar and spelling of text you wrote | Explain REP 105, then quiz me on it. Why does this NumPy cell raise a shape error? |
| Yellow: only after Pass 1, log it, verify it | Reviewing your requirements, trade study, or failure list; checking a calculation; asking what you might have missed | Here is my failure list. Ask me questions about modes I might be missing; do not rewrite the list |
| Red: not allowed | Choosing your scenario; writing your requirements, architecture, trade study weights or scores, failure analysis, or any text of the report or slides; drawing your figures or diagrams; producing numbers you do not check; any help during the questions after your talk | |

## Prompts that keep you thinking

The most useful prompts ask the tool to question you, not to answer for you.
<ul>
    <li> Act as a design reviewer. Ask me five questions about my requirements table, one at a time, without suggesting answers. </li>
    <li> Here is my stopping-distance calculation. Check each step and point to any error, without changing my design. </li>
    <li> Explain the difference between precision and accuracy with a GNSS example, then give me two questions to test myself. </li>
    <li> Find the official datasheet page for this part. Give me the link only. </li>
</ul>

## Verify everything

AI tools state wrong numbers with confidence. Common errors in this subject: sensor specifications that are invented or mixed up, such as the color camera's field of view given as the depth camera's; part numbers that do not exist; prices from years ago; ROS 1 packages presented as ROS 2 packages; and packages that were never released for Jazzy. Every number you use must trace to a primary source: a datasheet, a standard, a paper, or your own data. If you cannot find the source, do not use the number.

## The AI use log (Appendix B)

| # | Tool and version | Part | What you asked, in short | What you used | How you verified it | What you rejected, and why |
|---|---|---|---|---|---|---|
| 1 | | | | | | |

Mark every requirement, failure mode, or other item that came from an AI review, in the report as well as in the log. End the log with this statement, signed: *I used AI tools only as listed in this log. The design decisions, numbers, and text in this report are my own, and I can explain each of them.* If you used no AI tools, write None in the log and sign the statement anyway.

Disclosed green and yellow use costs you nothing. Undisclosed use, or any red-zone use, is an academic integrity matter.

## The design journal

`design_journal.md` holds plain notes, not polished text: your answers to the Pass 1 questions, photos of sketches, numbers you tried, and dead ends. Keep the dead ends, because they show your reasoning. A format that works:

```markdown
## Part 3, Pass 1
- Needs: detect a 0.3 m object at 0.6 m (Tool 1); HFOV of 64 deg or more (Tool 2)
- Candidates I know of: ...
- What worries me: ...
- Committed before any AI review
```

# What to submit

**On Canvas:**

1. Your report as one PDF, `[NortheasternUsername]_SDE_report.pdf`
2. Your slides as one PDF, `[NortheasternUsername]_SDE_slides.pdf`, before your talk

**In your repository:**

```
EECE5554/SDE/
├── README.md                 your scenario and a one-paragraph summary
├── design_journal.md         Pass 1 notes for every Part, committed before any AI review
├── analysis/
│   ├── sde_analysis.ipynb    your copy of this notebook, run with your numbers
│   └── figures/              every figure in your report and slides
├── report/                   the report source and its PDF
└── slides/                   the slides PDF
```

Commit as you go. The history of `design_journal.md` is how you show that your Pass 1 came first.

# Assessment (100 points)

The report is worth 70 points, the talk 20, and the process 10. Partial credit is available within each item, and the Canvas rubric shows the exact ratings.

**1. Problem Definition (10 pts)**
<ul>
    <li> Operational concept with the users, the operating envelope, and at least two off-nominal events (3 pts) </li>
    <li> Requirements table: at least eight, including the given ones and at least three derived ones with the math shown; values with units and conditions; a rationale and a verification method for each (5 pts) </li>
    <li> Assumptions and scope, stated plainly (2 pts) </li>
</ul>

**2. System Architecture (10 pts)**
<ul>
    <li> Functional architecture and the end-to-end pipeline figure, with rates (4 pts) </li>
    <li> Physical architecture: computers, buses, power, and where each function runs, with the safety functions independent of the main computer (3 pts) </li>
    <li> Operating modes with entry and exit conditions in numbers, including a degraded mode and a safe stop (3 pts) </li>
</ul>

**3. Sensor Selection (12 pts)**
<ul>
    <li> Sensing needs table derived from the requirements, with numbers (3 pts) </li>
    <li> At least two trade studies with at least three candidates each: must-meet screen, weights with reasons, cited datasheet numbers, and the sensitivity check (5 pts) </li>
    <li> Final sensor suite with placement, fields of view, and blind zones (2 pts) </li>
    <li> Prototype parts list from Tool 9: model, quantity, price, mass, power, source, and course-owned items (2 pts) </li>
</ul>

**4. Localization, Mapping & Navigation (14 pts)**
<ul>
    <li> Methods for localization, mapping, and navigation, each with one alternative and the reason it lost (4 pts) </li>
    <li> Estimator design: states, inputs, measurements with rates and noise, frames, and outlier rejection (3 pts) </li>
    <li> Error budget showing that the accuracy requirement is met, or by how much it is missed and what you changed (4 pts) </li>
    <li> Drift or outage analysis: how long the system can go without an absolute fix, and how the design handles longer gaps (3 pts) </li>
</ul>

**5. System Integration (10 pts)**
<ul>
    <li> TF tree, calibration plan, and time synchronization (3 pts) </li>
    <li> Data rate, compute, and power and runtime budgets, with margins (4 pts) </li>
    <li> Verification plan for the key requirements, and a traceability check with no open problems (3 pts) </li>
</ul>

**6. Failure Analysis (14 pts)**
<ul>
    <li> At least ten failure modes across sensing, estimation, computing, power, and the environment, scored and ranked (5 pts) </li>
    <li> A detection signal and threshold for each of the top five and for every severity 9 or 10 failure, not relying on the failed part (3 pts) </li>
    <li> Recovery: degraded modes and safe behavior for those failures (4 pts) </li>
    <li> Residual risk and open questions (2 pts) </li>
</ul>

**7. The five-slide talk (20 pts)**
<ul>
    <li> Five slides with the required content (8 pts) </li>
    <li> Clarity: claim titles, a figure or a table on every slide, units, readable text (4 pts) </li>
    <li> Timing: 3 pts for 5:00 or less, 1 pt for up to 5:30 (3 pts) </li>
    <li> Questions: answers about your own design, without notes, with reasons and numbers (5 pts) </li>
</ul>

**8. Process and documentation (10 pts)**
<ul>
    <li> Appendix A, tools used: complete and specific (2 pts) </li>
    <li> Appendix B, AI use log and the signed statement (3 pts) </li>
    <li> Design journal with Pass 1 notes committed before any AI review, and the analysis notebook run with your own numbers (3 pts) </li>
    <li> Report quality: numbered and captioned figures, cited sources, and a unit and a source for every number (2 pts) </li>
</ul>

**One deduction, as in Lab 2:** treating a precision number as an accuracy number in the error budget costs 2 points in Section 4.

### If you are stuck

Post on `[[ HELP_CHANNEL ]]` with a question about the method or a tool, such as what an input to Tool 5 means. Bring questions about your own design decisions to office hours, where we can talk through your reasoning without handing you an answer.

# Common pitfalls

<ul>
    <li> Requirements that name a solution, such as shall use a lidar, instead of a need </li>
    <li> Choosing the sensor first, then writing requirements that it happens to meet </li>
    <li> Numbers with no unit, no source, or no confidence level </li>
    <li> Datasheet values used outside their conditions: a lidar range quoted for a white target, used for a black one </li>
    <li> Trade study weights picked after the scores, so the favorite wins. The sensitivity check exposes this </li>
    <li> An error budget that lists only sensor noise and forgets calibration, time stamps, the map, and path tracking </li>
    <li> Runtime computed on a new pack on a mild day, with the computer and the sensors left out </li>
    <li> A failure list that says sensor fails, instead of specific modes such as stuck, biased, late, or dropped </li>
    <li> No safe state, or a safe state that depends on the main computer working </li>
    <li> Two redundant sensors that fail under the same condition </li>
    <li> A design copied from a famous robot without asking whether its requirements match yours </li>
    <li> More than five slides, or a talk that runs past five minutes </li>
</ul>

# Before you submit: questions a reviewer will ask

Answer each one for your own design. If one has no good answer, that is the part to fix.

1. Could a stranger test each requirement with only your table?
2. Where is the math for each derived requirement?
3. What is the worst operating condition you designed for, and why that one?
4. Which block in your pipeline handles each off-nominal event in your operational concept?
5. Which requirement made you choose each sensor, and which change in the requirements would change your choice?
6. Would your winner still win if your two largest weights were swapped?
7. Under which conditions does each chosen sensor fail?
8. What is your largest error source, and what would it cost to halve it?
9. How long can the system go without an absolute fix, and does the mission ever exceed that?
10. Is each number in your error budget precision or accuracy?
11. Which clock stamps each measurement, and how many meters is its error at top speed?
12. What is the runtime on an old pack on the coldest day?
13. Which link or bus is closest to its limit?
14. Which single failure could hurt someone, and how is it detected without the part that failed?
15. What does the system do in each degraded mode, and how does it leave it?
16. Which safety functions still work when the main computer crashes?
17. What is the smallest prototype that would test your riskiest assumption?
18. Which tests from your verification plan could you run in the final project with course hardware?
19. What did you change after a budget or an analysis failed?
20. If the client doubled one requirement tomorrow, which part of your design would break first?

# References

All are free to read.
<ul>
    <li> NASA Systems Engineering Handbook, NASA/SP-2016-6105 Rev 2: https://ntrs.nasa.gov/citations/20170001761. Section 4.2, Technical Requirements Definition, is the most useful part here </li>
    <li> REP 103, standard units and coordinate conventions: https://www.ros.org/reps/rep-0103.html </li>
    <li> REP 105, coordinate frames for mobile platforms: https://www.ros.org/reps/rep-0105.html </li>
    <li> robot_localization: https://github.com/cra-ros-pkg/robot_localization </li>
    <li> Nav2 documentation: https://docs.nav2.org </li>
    <li> TurtleBot 4 user manual: https://turtlebot.github.io/turtlebot4-user-manual/ </li>
    <li> Course materials: Labs 1 to 3, and Lecture 06 for the power budget </li>
</ul>

# Academic integrity

This exercise is individual work. You may use course materials, datasheets, documentation, and published papers, with citations. AI tools may be used only as described in *Using AI tools*, and every use is logged. Your requirements, design decisions, analysis, figures, report, and slides must be your own. A design that matches another student's, the worked example's, or an AI tool's output beyond coincidence will be treated as a violation.